# Analyse in Präsentationsreihenfolge

Datenvorbereitung und gemeinsam genutzte Hilfsfunktionen stehen am Anfang. Der Berechnungs- und Plot-Code folgt direkt bei den zugehörigen Abbildungen. Gespeicherte VAE-Ergebnisse und native AUTOENCODIX-Funktionen werden wiederverwendet.


## Daten und gemeinsame Darstellung


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

# Run this notebook with the repository root as the working directory.
ROOT = Path.cwd()

import json
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from autoencodix.evaluate._general_evaluator import GeneralEvaluator

RESULTS = ROOT / 'beta_vae_experiments/results_26_09'
OUTPUT = RESULTS / 'presentation_diagrams/regenerated'
OUTPUT.mkdir(parents=True, exist_ok=True)


def show_figures(*filenames):
    for filename in filenames:
        display(Image(filename=str(OUTPUT / filename), width=1100))


In [ ]:
def load_plot_data(results_dir, output_dir):
    """Load saved sweep tables and reproduce the original pooled-sex probe.

    No VAE training. Output paths are isolated from the original artifacts.
    """
    from pathlib import Path
    results_dir = Path(results_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    import json as _jw, numpy as np, pandas as pd, matplotlib
    import matplotlib.pyplot as plt
    from pathlib import Path
    from scipy.stats import spearmanr
    workspace_root = next((q for q in [output_dir,
        *output_dir.parents] if (q / 'autoencodix_package').is_dir()))
    sweep_dir = workspace_root / 'vae-beta-control/beta_vae_experiments/new_results_2026-09-29/biology_breakpoints_2026_09_29'
    output_dir.mkdir(exist_ok=True)
    clinical_metadata = pd.read_parquet(workspace_root / 'autoencodix_package/data/raw/OntixTutorialData/combined_clin_formatted.parquet')
    clinical_metadata.index = clinical_metadata.index.astype(str)
    COLS = {'auc_cancer': 'downstream_CANCER_TYPE_ACRONYM_z_valid_roc_auc_ovo',
        'rate': 'posterior_conditional_kl_total_mean',
        'au': 'posterior_active_units_count_eps_0p01',
        'collapsed': 'posterior_collapsed_dimensions_fraction_eps_0p01_delta_0p01',
        'tc': 'info_total_correlation_autoencodix',
        'coverage': 'latent_coverage_autoencodix_sensitivity', 'zdim': 'dimensionality_z_mle',
        'erank': 'dimensionality_mu_effective_rank', 'lcmc': 'geometry_mu_lcmc_zadu_k20',
        'trust': 'geometry_mu_trustworthiness_zadu_k20', 'mrre': 'geometry_mu_mrre_false_zadu_k20',
        'steadiness': 'geometry_mu_steadiness_zadu_k20',
        'continuity': 'geometry_mu_continuity_zadu_k20',
        'h1w': 'topology_h1_normalized_wasserstein',
        'edge': 'graph_directed_edge_jaccard_lcmc_related_k20'}
    run_metrics = pd.read_csv(sweep_dir / 'experiment_metrics.csv', usecols=['run_name',
        'latent_dim', 'seed', 'beta'] + list(COLS.values())).rename(columns={v: k for k,
        v in COLS.items()})
    run_metrics['beta_d'] = (run_metrics.beta / run_metrics.latent_dim).round(10)
    run_metrics = run_metrics.sort_values(['latent_dim', 'seed', 'beta_d']).reset_index(drop=True)
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    mixed_cancer_types = ['COAD', 'LUAD', 'LUSC', 'READ']
    sex_labels = clinical_metadata.SEX.astype(str)
    cancer_labels = clinical_metadata.CANCER_TYPE_ACRONYM.astype(str)

    def load_posterior(run, s):
        with np.load(sweep_dir / 'run_analysis' / run / 'posteriors' / f'posterior_{s}.npz') as a:
            return (a['mu'].astype(float), a['logvar'].astype(float), a['sample_ids'].astype(str))
    sex_auc_values = []
    for run in run_metrics.run_name:
        (mu_train, logvar_train, train_ids), (mu_valid, logvar_valid, valid_ids), (mu_test,
            logvar_test, test_ids) = (load_posterior(run, 'train'), load_posterior(run, 'valid'),
            load_posterior(run, 'test'))
        mu_evaluation, logvar_evaluation, evaluation_ids = (np.vstack([mu_valid, mu_test]),
            np.vstack([logvar_valid, logvar_test]), np.concatenate([valid_ids, test_ids]))
        known_train_mask = (sex_labels.reindex(train_ids).ne('unknown') & cancer_labels.reindex(train_ids).isin(mixed_cancer_types)).to_numpy()
        known_evaluation_mask = (sex_labels.reindex(evaluation_ids).ne('unknown') & cancer_labels.reindex(evaluation_ids).isin(mixed_cancer_types)).to_numpy()
        r = np.random.default_rng(21)
        train_features = mu_train[known_train_mask] + np.exp(0.5 * logvar_train[known_train_mask]) * r.standard_normal(mu_train[known_train_mask].shape)
        evaluation_features = mu_evaluation[known_evaluation_mask] + np.exp(0.5 * logvar_evaluation[known_evaluation_mask]) * r.standard_normal(mu_evaluation[known_evaluation_mask].shape)
        clf = LogisticRegression(max_iter=1000).fit(train_features,
            sex_labels.reindex(train_ids).to_numpy()[known_train_mask])
        yev = sex_labels.reindex(evaluation_ids).to_numpy()[known_evaluation_mask] == 'Male'
        sex_auc_values.append(roc_auc_score(yev, clf.predict_proba(evaluation_features)[:,
            list(clf.classes_).index('Male')]))
    run_metrics['auc_sex'] = sex_auc_values

    def normalise_endpoints(v):
        a, b = (v.iloc[0], v.iloc[-1])
        return (v - b) / (a - b) if abs(a - b) > 1e-12 else v * np.nan
    PROG = ['auc_cancer', 'auc_sex', 'tc', 'coverage', 'zdim', 'lcmc', 'trust', 'mrre',
        'steadiness', 'continuity', 'h1w', 'edge']
    for col in PROG:
        run_metrics['p_' + col] = run_metrics.groupby(['latent_dim',
            'seed'])[col].transform(normalise_endpoints)
    run_metrics['rate_rel'] = run_metrics.rate / run_metrics.groupby(['latent_dim',
        'seed']).rate.transform('first')
    beta_grid = sorted(run_metrics.beta_d.unique())
    beta_positions = np.arange(len(beta_grid))
    SUPW = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta(v):
        if v <= 0:
            return '0'
        m, e = f'{v:.2e}'.split('e')
        return f'{float(m):.3g}·10{str(int(e)).translate(SUPW)}'
    beta_labels = [format_beta(v) for v in beta_grid]

    def first_lasting_drop(p, thr=0.95):
        p = np.asarray(p, float)
        below = p < thr
        for i in range(len(p)):
            if below[i:].all():
                return i
        return np.nan
    NAVYW, BLUEW, ORANGEW, GREYW, GREENW, REDW, LIGHTW = ('#1B2430', '#2F6DB5', '#D9822B',
        '#9AA6B5', '#2E8B57', '#C0392B', '#DCE3EC')
    plot_style = {'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 15,
        'xtick.labelsize': 11, 'ytick.labelsize': 13, 'legend.fontsize': 12,
        'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
        'grid.color': '#E6E9EE', 'axes.edgecolor': '#5E6977'}

    def format_beta_axis(ax):
        ax.set_xticks(beta_positions)
        ax.set_xticklabels(beta_labels, rotation=45)
        ax.set_xlabel('β/d (compression →)')
    plot_summary = {}
    import json as _jt, numpy as np, pandas as pd, matplotlib
    import matplotlib.pyplot as plt
    from pathlib import Path
    workspace_root = next((q for q in [output_dir,
        *output_dir.parents] if (q / 'autoencodix_package').is_dir()))
    sweep_dir = workspace_root / 'vae-beta-control/beta_vae_experiments/new_results_2026-09-29/biology_breakpoints_2026_09_29'
    analysis_metrics = pd.read_csv(sweep_dir / 'experiment_metrics.csv')
    analysis_metrics = analysis_metrics[analysis_metrics.status.eq('ok')].copy()
    _n = analysis_metrics.select_dtypes('number').replace([np.inf, -np.inf], np.nan)
    analysis_metrics[_n.columns] = _n
    analysis_metrics = analysis_metrics.rename(columns={'posterior_conditional_kl_total_mean': 'R',
        'posterior_active_units_count_eps_0p01': 'AU',
        'posterior_collapsed_dimensions_fraction_eps_0p01_delta_0p01': 'CollapseFrac',
        'downstream_CANCER_TYPE_ACRONYM_z_valid_roc_auc_ovo': 'auc_cancer'})
    analysis_metrics['beta_d'] = (analysis_metrics.beta / analysis_metrics.latent_dim).round(10)
    analysis_metrics['logR'] = np.log(analysis_metrics.R.clip(lower=1e-12))
    SXT = pd.read_csv(results_dir / '30_09_G16_geschlecht_gemischt.csv').set_index('run_name').sex_mixed_z1
    analysis_metrics['auc_sex'] = analysis_metrics.run_name.map(SXT)
    RMINT = {}
    for dim, g in analysis_metrics[analysis_metrics.beta.gt(0)].groupby('latent_dim'):
        ons = []
        for _, t in g.groupby('seed'):
            t = t.sort_values('R', ascending=False)
            m = t.CollapseFrac >= 0.5
            ons.append(t.R[m].iloc[0] if m.any() else 0.0)
        RMINT[dim] = float(np.median(ons))
    admissible_states = analysis_metrics[analysis_metrics.beta.gt(0) & analysis_metrics.AU.gt(0) & (analysis_metrics.R >= analysis_metrics.latent_dim.map(RMINT))].sort_values(['latent_dim',
        'seed', 'beta']).reset_index(drop=True)
    PREFT = ('geometry_', 'graph_', 'dimensionality_', 'dtm_', 'topology_', 'info_')
    _alive = analysis_metrics.beta.gt(0) & analysis_metrics.AU.gt(0)
    _cand = [c for c in _n.columns if c.startswith(PREFT) and (not c.startswith(('graph_input_',
        'graph_delta_',
        'dimensionality_x_'))) and (not c.endswith('_x')) and (analysis_metrics.loc[_alive,
        c].notna().mean() >= 0.8) and (analysis_metrics.loc[_alive, c].nunique() >= 3)]
    usable_metrics = [c for c in _cand if admissible_states[c].notna().all() and admissible_states.groupby(['latent_dim',
        'seed'])[c].nunique().min() >= 4]
    NAMET = {'geometry_mu_steadiness_zadu_k20': 'Steadiness',
        'geometry_mu_scale_normalized_stress_zadu': 'Stress (scale-norm.)',
        'geometry_mu_non_metric_stress_zadu': 'Stress (non-metric)',
        'geometry_mu_trustworthiness_zadu_k20': 'Trustworthiness',
        'geometry_mu_continuity_zadu_k20': 'Continuity', 'geometry_mu_lcmc_zadu_k20': 'LCMC',
        'geometry_mu_mrre_false_zadu_k20': 'MRRE',
        'geometry_mu_mrre_missing_zadu_k20': 'MRRE (missing)',
        'graph_directed_edge_jaccard_lcmc_related_k20': 'Edge overlap',
        'graph_latent_clustering_k20': 'Graph clustering',
        'topology_h1_normalized_wasserstein': 'H1 Wasserstein',
        'topology_landscape_h1_l2': 'H1 landscape',
        'dimensionality_mu_effective_rank': 'Effective rank',
        'dimensionality_mu_effective_rank_fraction': 'Effective rank (frac.)',
        'dimensionality_z_mle': 'Intrinsic dim. of z',
        'info_total_correlation_autoencodix': 'Total correlation',
        'geometry_mu_cohesiveness_zadu_k20': 'Cohesiveness',
        'topology_h0_persistence_entropy_mu': 'H0 persistence entropy',
        'graph_latent_groupie_ratio_k20': 'Graph groupie ratio',
        'graph_latent_lambda2_k20': 'Graph connectivity (λ₂)',
        'graph_latent_transitivity_k20': 'Graph transitivity',
        'graph_latent_global_efficiency_k20': 'Graph global efficiency',
        'graph_latent_forman_std_k20': 'Graph curvature spread',
        'logR': 'Rate (log R)'}

    def metric_label(c):
        if c in NAMET:
            return NAMET[c]
        s = c.replace('geometry_mu_', '').replace('graph_latent_', 'graph ').replace('_zadu',
            '').replace('_k20', '').replace('dimensionality_mu_', '').replace('topology_',
            '').replace('info_', '')
        return s.replace('_', ' ')
    analysis_beta_grid = sorted(analysis_metrics.beta_d.unique())
    analysis_beta_positions = np.arange(len(analysis_beta_grid))
    SUPT = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta_label(v):
        if v != v:
            return '–'
        if v <= 0:
            return '0'
        m, e = f'{v:.2e}'.split('e')
        return f'{float(m):.3g}·10{str(int(e)).translate(SUPT)}'
    analysis_beta_labels = [format_beta_label(v) for v in analysis_beta_grid]
    NAVYT, BLUET, ORANGET, GREYT, GREENT, REDT, TEALT = ('#1B2430', '#2F6DB5', '#D9822B', '#9AA6B5',
        '#2E8B57', '#C0392B', '#00A6A6')
    analysis_plot_style = {'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 15,
        'xtick.labelsize': 11, 'ytick.labelsize': 13, 'legend.fontsize': 12,
        'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
        'grid.color': '#E6E9EE', 'axes.edgecolor': '#5E6977'}
    analysis_summary = {}
    hidden_metrics = {'graph_directed_edge_jaccard_lcmc_related_k20',
        'geometry_mu_mrre_false_zadu_k20', 'geometry_mu_mrre_missing_zadu_k20'}
    analysis_metrics['auc_sex_all'] = analysis_metrics.run_name.map(run_metrics.set_index('run_name').auc_sex)
    rate_correlations = pd.read_csv(results_dir / '30_09_B_rate_korrelation.csv').set_index('Metrik').rho_rate.to_dict()
    course_correlations = {key: pd.read_csv(results_dir / f'30_09_T7g_verlauf_{key}.csv',
        index_col=0) for key in ['cancer', 'sex']}
    return {
        'BLUET': BLUET,
        'BLUEW': BLUEW,
        'GREENT': GREENT,
        'GREENW': GREENW,
        'GREYW': GREYW,
        'LIGHTW': LIGHTW,
        'NAVYT': NAVYT,
        'NAVYW': NAVYW,
        'ORANGET': ORANGET,
        'ORANGEW': ORANGEW,
        'REDT': REDT,
        'REDW': REDW,
        'TEALT': TEALT,
        '_jw': _jw,
        'admissible_states': admissible_states,
        'analysis_metrics': analysis_metrics,
        'analysis_plot_style': analysis_plot_style,
        'analysis_summary': analysis_summary,
        'beta_grid': beta_grid,
        'beta_labels': beta_labels,
        'beta_positions': beta_positions,
        'clinical_metadata': clinical_metadata,
        'course_correlations': course_correlations,
        'first_lasting_drop': first_lasting_drop,
        'format_beta': format_beta,
        'format_beta_axis': format_beta_axis,
        'format_beta_label': format_beta_label,
        'hidden_metrics': hidden_metrics,
        'load_posterior': load_posterior,
        'metric_label': metric_label,
        'mixed_cancer_types': mixed_cancer_types,
        'output_dir': output_dir,
        'plot_style': plot_style,
        'plot_summary': plot_summary,
        'rate_correlations': rate_correlations,
        'results_dir': results_dir,
        'run_metrics': run_metrics,
        'sweep_dir': sweep_dir,
        'usable_metrics': usable_metrics,
        'workspace_root': workspace_root
    }

data = load_plot_data(RESULTS, OUTPUT)


## Folie 5 — 24_sex_by_type.png


In [ ]:
def plot_sex_distribution(data, output_dir):
    NAVYW = data['NAVYW']
    _jw = data['_jw']
    clinical_metadata = data['clinical_metadata']
    mixed_cancer_types = data['mixed_cancer_types']
    plot_style = data['plot_style']
    plot_summary = data['plot_summary']
    XT7 = pd.crosstab(clinical_metadata.CANCER_TYPE_ACRONYM.astype(str), clinical_metadata.SEX.astype(str)).reindex(columns=['Female', 'Male', 'unknown'], fill_value=0)
    XT7 = XT7.loc[XT7.sum(1).sort_values().index]
    with matplotlib.rc_context(plot_style):
        fig, ax = plt.subplots(figsize=(10, 5.6))
        y = np.arange(len(XT7))
        left = np.zeros(len(XT7))
        for col, c, l in [('Female', '#00A6A6', 'female'), ('Male', '#1A1A1A', 'male'), ('unknown', '#9AA6B5', 'unknown')]:
            ax.barh(y, XT7[col], left=left, color=c, label=l, height=0.65)
            left += XT7[col].values
        for yy, (k, r) in zip(y, XT7.iterrows()):
            ax.text(left[yy] + 15, yy, f'{r.Female} / {r.Male} / {r.unknown}', va='center', fontsize=12, color=NAVYW)
        ax.set_yticks(y)
        ax.set_yticklabels([f'{k}  (mixed)' if k in mixed_cancer_types else k for k in XT7.index])
        for t in ax.get_yticklabels():
            if 'mixed' in t.get_text():
                t.set_color('#00A6A6')
                t.set_fontweight('bold')
        ax.set_xlabel('samples (female / male / unknown)')
        ax.set_xlim(0, left.max() * 1.3)
        ax.legend(frameon=False, loc='lower right')
        ax.grid(axis='y', visible=False)
        fig.tight_layout()
        fig.savefig(output_dir / '24_sex_by_type.png', dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    m7 = XT7.loc[mixed_cancer_types]
    plot_summary['w7'] = {'mixed_f': int(m7.Female.sum()), 'mixed_m': int(m7.Male.sum()), 'mixed_u': int(m7.unknown.sum()), 'total': int(XT7.values.sum()), 'total_f': int(XT7.Female.sum()), 'total_m': int(XT7.Male.sum()), 'total_u': int(XT7.unknown.sum())}
    _jw.dump(plot_summary, open(output_dir / 'wtasks.json', 'w'), ensure_ascii=False, indent=1)


In [ ]:
plot_sex_distribution(data, OUTPUT)
show_figures('24_sex_by_type.png')


## Folie 7 — 00_main.png


In [ ]:
def plot_overview_main(data, output_dir):
    run_metrics = data['run_metrics']
    sweep_dir = data['sweep_dir']
    import matplotlib as mpl
    output_dir.mkdir(exist_ok=True)
    NAVY, BLUE, ORANGE, GREY, GREEN, RED, LIGHT = ('#1B2430', '#2F6DB5', '#D9822B', '#9AA6B5', '#2E8B57', '#C0392B', '#DCE3EC')
    overview_plot_style = {'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 15, 'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 12, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#E6E9EE', 'axes.edgecolor': '#5E6977', 'font.family': 'DejaVu Sans'}

    def save(fig, name):
        f = output_dir / f'{name}.png'
        fig.savefig(f, dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return f
    indexed_metrics = pd.read_csv(sweep_dir / 'experiment_metrics.csv').set_index('run_name')
    RATE = 'posterior_conditional_kl_total_mean'
    task_scores = run_metrics.rename(columns={'auc_cancer': 'AUC_valid', 'auc_sex': 'sex_mixed_z'})
    GRID = sorted(task_scores.beta_d.unique())
    pos_of = lambda v: np.nan if v != v else 0.0 if v <= 0 else float(np.interp(np.log10(v), np.log10(GRID[1:]), np.arange(1, len(GRID))))
    SUP = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta_short(v):
        if v != v:
            return '–'
        if v <= 0:
            return '0'
        m, e = f'{v:.2e}'.split('e')
        return f'{float(m):.2g}·10{str(int(e)).translate(SUP)}'
    gridlab = [format_beta_short(b) for b in GRID]
    FIGS = {}
    overview_summary = {}
    with mpl.rc_context(overview_plot_style):
        M0 = task_scores[task_scores.latent_dim.ge(4)].copy()
        M0['R'] = M0.run_name.map(indexed_metrics[RATE])
        M0['recon'] = M0.run_name.map(indexed_metrics['recon_val_mse'])
        g0 = M0.groupby('beta_d')[['R', 'recon', 'AUC_valid', 'sex_mixed_z']].median()
        xs = [pos_of(b) for b in g0.index]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
        axes[0].plot(xs, g0.R, color=NAVY, lw=3, marker='o')
        axes[0].set_yscale('log')
        axes[0].set_title('Rate R [nats]')
        axes[1].plot(xs, g0.recon, color=ORANGE, lw=3, marker='o')
        axes[1].set_title('Reconstruction error (MSE, validation)')
        axes[2].plot(xs, g0.AUC_valid, color=BLUE, lw=3, marker='o', label='cancer type')
        axes[2].plot(xs, g0.sex_mixed_z, color=GREEN, lw=3, marker='o', label='sex (mixed types)')
        axes[2].axhline(0.5, color=GREY, lw=1)
        axes[2].set_ylim(0.4, 1.02)
        axes[2].set_title('Downstream AUC on z')
        axes[2].legend(frameon=False, loc='center left', bbox_to_anchor=(0, 0.42))
        for ax in axes:
            ax.set_xticks(range(len(GRID)))
            ax.set_xticklabels(gridlab, fontsize=10, rotation=45)
            ax.set_xlabel('β/d (compression →)')
        fig.tight_layout()
        FIGS['main'] = save(fig, '00_main')
        overview_summary['R0'] = g0.R.iloc[0]
        overview_summary['R_1e3'] = g0.R.loc[0.001]
        overview_summary['rec0'] = g0.recon.iloc[0]
        overview_summary['rec_1e3'] = g0.recon.loc[0.001]
        overview_summary['rec_end'] = g0.recon.iloc[-1]
        overview_summary['can_1e3'] = g0.AUC_valid.loc[0.001]
        overview_summary['sex_1e3'] = g0.sex_mixed_z.loc[0.001]


In [ ]:
plot_overview_main(data, OUTPUT)
show_figures('00_main.png')


### Folie 7 — zusätzliche Übersicht aller Downstream-Aufgaben

Native AUTOENCODIX-Probes auf einem reproduzierbaren z-Sample; Training auf Train, Darstellung der Validierungs-AUC. Median und Interquartilsabstand über d ≥ 4.


#### Aufgaben und native Klassifikations-Probes

Die gemeinsamen Lade- und Auswahlfunktionen werden auch für die späteren Klassenstruktur-Plots verwendet.


In [ ]:
TASKS = {
    'cancer_type': ('CANCER_TYPE_ACRONYM', None),
    'cancer_groups': ('CANCER_TYPE', None),
    'brca_subtypes': ('SUBTYPE', ['BRCA']),
    'ucec_subtypes': ('SUBTYPE', ['UCEC']),
    'detailed_histology': ('CANCER_TYPE_DETAILED', None),
    'stage_I_IV': ('AJCC_PATHOLOGIC_TUMOR_STAGE_SHORT', None),
    'sex_all': ('SEX', None),
    'sex_mixed': ('SEX', ['COAD', 'LUAD', 'LUSC', 'READ']),
}

def load_latent_frames(sweep_dir, clinical_metadata, run_name):
    """Load the same seed-31 posterior draw used by the native AUC probes."""
    sweep_dir = Path(sweep_dir)
    rng = np.random.default_rng(31)
    frames = {}
    for split in ['train', 'valid', 'test']:
        posterior = sweep_dir / 'run_analysis' / run_name / 'posteriors' / f'posterior_{split}.npz'
        with np.load(posterior, allow_pickle=False) as arrays:
            mu, logvar = arrays['mu'].astype(float), arrays['logvar'].astype(float)
            ids = arrays['sample_ids'].astype(str)
        if mu.shape != logvar.shape or len(ids) != len(mu) or len(set(ids)) != len(ids):
            raise ValueError(f'Invalid posterior shapes/IDs: {posterior}')
        if not pd.Index(ids).isin(clinical_metadata.index).all():
            raise ValueError(f'Missing clinical IDs: {posterior}')
        if not np.isfinite(mu).all() or not np.isfinite(logvar).all():
            raise ValueError(f'Nonfinite posterior: {posterior}')
        z = mu + np.exp(.5 * logvar) * rng.standard_normal(mu.shape)
        frames[split] = pd.DataFrame(z, index=ids)
    split_ids = [set(frame.index) for frame in frames.values()]
    if any(split_ids[i] & split_ids[j] for i in range(3) for j in range(i)):
        raise ValueError('Overlapping data splits')
    return frames

def select_task_frames(frames, clinical_metadata, target, cohort):
    """Use known labels and classes with >=20 TRAIN samples, exactly as the probes."""
    selected = {}
    for split, frame in frames.items():
        annotations = clinical_metadata.reindex(frame.index)
        labels = annotations[target].astype('string').str.strip()
        keep = labels.notna() & ~labels.str.lower().isin(['unknown', '', 'nan', 'none'])
        if cohort:
            keep &= annotations.CANCER_TYPE_ACRONYM.isin(cohort)
        selected[split] = frame.loc[keep]
    counts = clinical_metadata.loc[selected['train'].index, target].value_counts()
    classes = set(counts[counts >= 20].index)
    if len(classes) < 2:
        raise ValueError(f'Fewer than two training classes: {target}')
    selected = {split: frame.loc[clinical_metadata.loc[frame.index, target].isin(classes)]
                for split, frame in selected.items()}
    return selected, classes


In [ ]:
def evaluate_tasks(sweep_dir, clinical_metadata, output_dir):
    """Fit on train, report native macro-OvO AUC on valid and test.

    Matches the additional notebook: one z draw, seed 31, minimum 20
    training samples per class. Missing evaluation classes remain NaN.
    Each completed run is cached to allow interrupted sweeps to resume.
    """
    sweep_dir, output_dir = Path(sweep_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    runs = pd.read_csv(sweep_dir / 'experiment_metrics.csv')
    runs = runs[runs.status.eq('ok')].copy()
    cache = output_dir / 'native_task_cache'
    cache.mkdir(exist_ok=True)
    results = []
    for number, run in enumerate(runs.itertuples(), 1):
        path = cache / f'{run.run_name}.parquet'
        if path.exists():
            cached = pd.read_parquet(path)
            if "probe_max_iter" in cached:
                results.append(cached)
                continue
        frames = load_latent_frames(sweep_dir, clinical_metadata, run.run_name)
        scores = []
        for task, (target, cohort) in TASKS.items():
            selected, classes = select_task_frames(frames, clinical_metadata, target, cohort)
            eligible = {'train': selected['train']}
            missing = []
            for split in ['valid', 'test']:
                if set(clinical_metadata.loc[selected[split].index, target]) == classes:
                    eligible[split] = selected[split]
                else:
                    missing.append({'score_split': split, 'CLINIC_PARAM': target,
                                    'metric': 'roc_auc_ovo', 'value': np.nan,
                                    'status': 'missing_evaluation_class'})
            sample_split = pd.DataFrame([{'SAMPLE_ID': sid, 'SPLIT': split}
                                         for split, frame in eligible.items() for sid in frame.index])
            for max_iter in [1000, 10000]:
                with warnings.catch_warnings(record=True) as recorded:
                    warnings.simplefilter('always', ConvergenceWarning)
                    native = GeneralEvaluator._single_ml_presplit(
                        sample_split=sample_split, df=pd.concat(eligible.values()),
                        clin_data=clinical_metadata, task_param=target,
                        sklearn_ml=LogisticRegression(max_iter=max_iter),
                        metric='roc_auc_ovo', ml_type='classification', top_k_classes=None)
                converged = not any(issubclass(w.category, ConvergenceWarning) for w in recorded)
                if converged:
                    break
            native['probe_max_iter'] = max_iter
            native['status'] = 'ok' if converged else 'probe_not_converged'
            if not converged:
                native['value'] = np.nan
            if missing:
                native = pd.concat([native, pd.DataFrame(missing)], ignore_index=True)
            scores.append(native.assign(task=task, run_name=run.run_name,
                                        latent_dim=run.latent_dim, seed=run.seed,
                                        beta=run.beta, beta_per_dim=round(run.beta / run.latent_dim, 10),
                                        representation='z_redrawn', random_state=31,
                                        min_train_class=20))
        result = pd.concat(scores, ignore_index=True)
        result.to_parquet(path, index=False)
        results.append(result)
        print(f'Native downstream tasks: {number}/{len(runs)} models', flush=True)
    combined = pd.concat(results, ignore_index=True)
    combined.to_parquet(output_dir / 'downstream_all_tasks.parquet', index=False)
    return combined


In [ ]:
def plot_all_downstream_tasks(data, output_dir):
    """Additional slide-7 plot: native task AUCs over the saved beta sweep.

    Validation curves use d >= 4, matching the overview population.
    Shading shows the interquartile range across 20 trajectories, not a CI.
    """

    scores = evaluate_tasks(data['sweep_dir'], data['clinical_metadata'], output_dir)
    validation = scores[(scores.score_split == 'valid') & (scores.latent_dim >= 4)]
    titles = {
        'cancer_type': 'Cancer type', 'cancer_groups': 'Cancer groups',
        'brca_subtypes': 'BRCA molecular subtype', 'ucec_subtypes': 'UCEC molecular subtype',
        'detailed_histology': 'Detailed histology', 'stage_I_IV': 'Tumour stage (I–IV)',
        'sex_all': 'Sex · all cancer types', 'sex_mixed': 'Sex · mixed cancer types',
    }
    beta_grid = data['beta_grid']
    positions = data['beta_positions']
    with matplotlib.rc_context(data['plot_style']):
        figure, axes = plt.subplots(2, 4, figsize=(23, 10), sharex=True, sharey=True)
        for axis, (task, title) in zip(axes.flat, titles.items()):
            task_scores = validation[validation.task == task]
            grouped = task_scores.groupby('beta_per_dim').value
            median = grouped.median().reindex(beta_grid)
            lower = grouped.quantile(.25).reindex(beta_grid)
            upper = grouped.quantile(.75).reindex(beta_grid)
            count = grouped.count().reindex(beta_grid, fill_value=0)
            axis.fill_between(positions, lower, upper, color='#2F6DB5', alpha=.15)
            axis.plot(positions, median, color='#2F6DB5', marker='o', linewidth=2.5)
            axis.axhline(.5, color='#9AA6B5', linestyle=':', linewidth=1)
            count_label = str(count.min()) if count.min() == count.max() else f'{count.min()}–{count.max()}'
            axis.set_title(f'{title}\nn = {count_label} trajectories / β')
            axis.set_ylim(.4, 1.02)
            data['format_beta_axis'](axis)
            axis.tick_params(axis='x', labelbottom=True)
        for axis in axes[:, 0]:
            axis.set_ylabel('Validation macro-OvO AUC')
        figure.suptitle('All downstream tasks · native AUTOENCODIX logistic regression', fontsize=20)
        figure.text(.5, .01, 'Median and interquartile range · d ≥ 4 · one posterior z draw (seed 31) · '
                    'subtype tasks use their cancer cohort · gaps indicate missing evaluation classes',
                    ha='center', fontsize=12)
        figure.tight_layout(rect=(0, .045, 1, .95))
        figure.savefig(output_dir / '00_all_downstream_tasks.png', dpi=200,
                       bbox_inches='tight', facecolor='white')
        plt.close(figure)
    return scores


In [ ]:
all_task_scores = plot_all_downstream_tasks(data, OUTPUT)
show_figures('00_all_downstream_tasks.png')


## Folie 9 — 27_vae_vs_pca.png


In [ ]:
def plot_pca_comparison(data, output_dir):
    """PCA fitted to train, compared with stored VAE reconstruction metrics."""
    BLUEW = data['BLUEW']
    GREYW = data['GREYW']
    load_posterior = data['load_posterior']
    plot_style = data['plot_style']
    run_metrics = data['run_metrics']
    sweep_dir = data['sweep_dir']
    workspace_root = data['workspace_root']
    from sklearn.decomposition import PCA
    from sklearn.metrics import explained_variance_score
    XSP = workspace_root / 'vae-beta-control/archive_06_10_26/results/biology_no_annealing_2026-09-28/biology/input_data/x_splits_split_seed2.npz'
    with np.load(XSP, allow_pickle=False) as a:
        train_features, test_features = (a['x_train'].astype(float), a['x_test'].astype(float))
        reference_ids = a['ids_test'].astype(str)
    if set(reference_ids) != set(load_posterior(run_metrics.run_name.iloc[0], 'test')[2]):
        raise ValueError('PCA input/posterior split mismatch')
    EVc = pd.read_csv(sweep_dir / 'experiment_metrics.csv', usecols=['run_name',
        'recon_test_explained_variance_uniform_average', 'recon_test_mse'])
    MV = run_metrics.merge(EVc, on='run_name')
    MV['EV_varweighted'] = 1 - MV.recon_test_mse / test_features.var(0).mean()
    T1 = MV[MV.beta_d.eq(0)].groupby('latent_dim')[['recon_test_explained_variance_uniform_average',
        'EV_varweighted']].median()
    T1.columns = ['EV (gleich gewichtet)', 'EV (varianzgewichtet)']
    from sklearn.decomposition import PCA
    from sklearn.metrics import explained_variance_score
    pca = PCA(n_components=200, random_state=0).fit(train_features)
    Ste = pca.transform(test_features)
    rowsP = []
    for k in [2, 4, 6, 8, 12, 16, 32, 64, 128]:
        Xh = Ste[:, :k] @ pca.components_[:k] + pca.mean_
        rowsP.append(dict(k=k, EV_uniform=explained_variance_score(test_features, Xh,
            multioutput='uniform_average'), EV_varweighted=explained_variance_score(test_features,
            Xh, multioutput='variance_weighted')))
    PCAt = pd.DataFrame(rowsP).set_index('k')
    cmp = pd.DataFrame({'VAE EV gleich (β/d=0)': T1['EV (gleich gewichtet)'],
        'PCA EV gleich (k=d)': PCAt.EV_uniform.reindex(T1.index),
        'VAE EV varianzgew.': T1['EV (varianzgewichtet)'],
        'PCA EV varianzgew. (k=d)': PCAt.EV_varweighted.reindex(T1.index)})
    B0 = MV[MV.beta_d == 0]
    with matplotlib.rc_context(plot_style):
        fig, ax = plt.subplots(1, 2, figsize=(15, 5.6), sharey=False)
        for a_, pc, vc, tt in [(ax[0], 'EV_uniform',
            'recon_test_explained_variance_uniform_average', 'equal weight per feature'), (ax[1],
            'EV_varweighted', 'EV_varweighted', 'weighted by feature variance')]:
            a_.plot(PCAt.index, PCAt[pc], color=GREYW, lw=2.5, marker='s',
                label='PCA with k components')
            g = B0.groupby('latent_dim')[vc]
            a_.errorbar(g.median().index, g.median(), yerr=[g.median() - g.quantile(0.25),
                g.quantile(0.75) - g.median()], color=BLUEW, lw=2.5, marker='o', ms=8, capsize=4,
                label='VAE, β/d = 0 (d = k)')
            a_.set_xscale('log', base=2)
            a_.set_xticks([2, 4, 8, 16, 32, 64, 128])
            a_.set_xticklabels(['2', '4', '8', '16', '32', '64', '128'])
            a_.set_xlabel('latent dimensions d / PCA components k')
            a_.set_title('explained variance, test split · ' + tt)
            a_.set_ylim(0, 0.8)
        ax[0].legend(frameon=False, loc='upper left')
        fig.tight_layout()
        fig.savefig(output_dir / '27_vae_vs_pca.png', dpi=200, bbox_inches='tight',
            facecolor='white')
        plt.close(fig)


In [ ]:
plot_pca_comparison(data, OUTPUT)
show_figures('27_vae_vs_pca.png')


## Folie 10 — 16_latent2d_best.png


In [ ]:
def _native_latent_panel(embedding, labels, palette, title, title_color='#2F6DB5'):
    """Native AUTOENCODIX scatter and centres, with shared presentation styling."""
    from matplotlib.colors import to_rgba
    from autoencodix.visualize import GeneralVisualizer

    figure = GeneralVisualizer._plot_2D(
        embedding=embedding, labels=labels, center=True,
        layer='', figsize=(9, 7), no_leg=True, focus_labels=sorted(set(labels)),
    )
    figure.set_size_inches(9, 7)
    axis = figure.axes[0]
    points, centres = axis.collections[:2]
    points.set_facecolors([to_rgba(palette[label]) for label in labels])
    points.set_edgecolors('none')
    points.set_sizes([7])
    points.set_alpha(0.75)

    # AUTOENCODIX sorts centre labels; hide centres of background cohorts.
    centre_labels = sorted(set(labels))
    visible = np.array([label not in {'other', 'unknown'} for label in centre_labels])
    centres.set_offsets(centres.get_offsets()[visible])
    centres.set_facecolors([to_rgba(palette[label])
                           for label in centre_labels if label not in {'other', 'unknown'}])
    centres.set_sizes([520])
    centres.set_edgecolors('black')
    centres.set_alpha(1)
    axis.set_title(title, fontsize=14, color='white', fontweight='bold',
                          pad=10, bbox=dict(boxstyle='round,pad=.35',
                                           fc=title_color, ec='none'))
    axis.set(xlabel='', ylabel='', xticks=[], yticks=[])
    axis.grid(False)
    for spine in axis.spines.values():
        spine.set_color('#C9CED6')
    figure.tight_layout()

    return figure


In [ ]:
def prepare_latent_views(data, output_dir):
    """Seven original latent-space views using cached UMAP coordinates."""
    clinical_metadata = data['clinical_metadata']
    load_posterior = data['load_posterior']
    results_dir = data['results_dir']
    run_metrics = data['run_metrics']
    sweep_dir = data['sweep_dir']
    runs = run_metrics[['run_name', 'latent_dim', 'seed', 'beta', 'beta_d']].copy()
    embeddings = {}
    for run in runs.run_name:
        path = sweep_dir / 'plots' / run / '2D_embedding_z_all.csv'
        if not path.exists():
            raise FileNotFoundError(path)
        e = pd.read_csv(path, index_col=0)
        e.index = e.index.astype(str)
        expected = np.concatenate([load_posterior(run, s)[2] for s in ['train', 'valid', 'test']])
        if not e.index.is_unique or set(e.index) != set(expected):
            raise ValueError('UMAP/posterior ID mismatch')
        if e.shape[1] != 2 or not np.isfinite(e.to_numpy()).all():
            raise ValueError('Invalid UMAP coordinates')
        embeddings[run] = (pd.DataFrame(e.to_numpy(), columns=[0, 1]), e.index.to_numpy())
    output_dir.mkdir(exist_ok=True)
    task_scores = pd.read_csv(sweep_dir / 'experiment_metrics.csv', usecols=['run_name', 'downstream_CANCER_TYPE_ACRONYM_z_valid_roc_auc_ovo']).rename(columns={'downstream_CANCER_TYPE_ACRONYM_z_valid_roc_auc_ovo': 'auc_cancer'})
    task_scores = task_scores.merge(pd.read_csv(results_dir / '30_09_G16_geschlecht_gemischt.csv', usecols=['run_name', 'sex_mixed_z1']).rename(columns={'sex_mixed_z1': 'auc_sex'}), on='run_name').merge(runs, on='run_name')
    mixed_cancer_types = ['COAD', 'LUAD', 'LUSC', 'READ']
    cancer_types = sorted(clinical_metadata.CANCER_TYPE_ACRONYM.astype(str).unique())
    cancer_colors = plt.get_cmap('tab10').colors
    superscript_digits = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta_per_dim(v):
        if v <= 0:
            return '0'
        mm, e = f'{v:.2e}'.split('e')
        return f'{float(mm):.3g}·10{str(int(e)).translate(superscript_digits)}'

    def select_run(col, collapse_of=None):
        if collapse_of is None:
            r = task_scores.loc[task_scores[col].idxmax()]
        else:
            sub = task_scores[(task_scores.latent_dim == collapse_of.latent_dim) & (task_scores.seed == collapse_of.seed)]
            r = sub.loc[sub.beta_d.idxmax()]
        return r
    selected_runs = {'best': {'cancer': select_run('auc_cancer'), 'sex': select_run('auc_sex')}}
    cancer_preserved = task_scores[task_scores.auc_cancer >= 0.95]
    split_run = cancer_preserved.loc[cancer_preserved.auc_sex.idxmin()]
    selected_runs['split'] = {'cancer': split_run, 'sex': split_run}
    selected_runs['collapse'] = {'cancer': select_run('auc_cancer', selected_runs['best']['cancer']), 'sex': select_run('auc_sex', selected_runs['best']['sex'])}
    best_cancer_run, best_sex_run = (selected_runs['best']['cancer'], selected_runs['best']['sex'])
    selected_runs['best_cmodel'] = {'cancer': best_cancer_run, 'sex': best_cancer_run}
    selected_runs['best_smodel'] = {'cancer': best_sex_run, 'sex': best_sex_run}
    selected_runs['coll_cmodel'] = {'cancer': selected_runs['collapse']['cancer'], 'sex': selected_runs['collapse']['cancer']}
    selected_runs['coll_smodel'] = {'cancer': selected_runs['collapse']['sex'], 'sex': selected_runs['collapse']['sex']}

    def panel(ax, run, task):
        """Use AUTOENCODIX for points and class centres; only adapt slide styling."""
        import io
        embedding, sample_ids = embeddings[run.run_name]
        annotations = clinical_metadata.reindex(sample_ids)
        cancer = annotations.CANCER_TYPE_ACRONYM.astype(str)
        sex = annotations.SEX.astype(str)
        cancer_palette = dict(zip(cancer_types, cancer_colors))
        if task == 'cancer':
            labels = cancer.tolist()
            palette = cancer_palette
            title = f'TASK 1 · CANCER TYPE  ·  AUC {run.auc_cancer:.3f}'
            title_color = '#2F6DB5'
        else:
            labels = [f'{c} / {s}' if c in mixed_cancer_types and s in ['Female', 'Male'] else 'unknown' if c in mixed_cancer_types and s == 'unknown' else 'other' for c, s in zip(cancer, sex)]
            palette = {label: '#00A6A6' if label.endswith(' / Female') else '#1A1A1A' for label in labels}
            palette.update(other='#DCE3EC', unknown='#9AA6B5')
            title = f'TASK 2 · SEX (mixed cancer types)  ·  AUC {run.auc_sex:.3f}'
            title_color = '#00A6A6'
        native_figure = _native_latent_panel(embedding, labels, palette, title, title_color)
        with io.BytesIO() as buffer:
            native_figure.savefig(buffer, format='png', dpi=200, bbox_inches='tight')
            buffer.seek(0)
            ax.imshow(plt.imread(buffer))
        plt.close(native_figure)
        ax.axis('off')
    plot_metadata = {}

    def draw_latent_view(key, filename):
        fig, axes = plt.subplots(1, 2, figsize=(18, 7.6))
        panel(axes[0], selected_runs[key]['cancer'], 'cancer')
        panel(axes[1], selected_runs[key]['sex'], 'sex')
        cancer_legend = [plt.Line2D([], [], marker='o', ls='', color=cancer_colors[i], label=c) for i, c in enumerate(cancer_types)]
        sex_legend = [plt.Line2D([], [], marker='o', ls='', color=c, label=l) for l, c in [('female', '#00A6A6'), ('male', '#1A1A1A'), ('sex unknown (mixed types)', '#9AA6B5'), ('single-sex cancer types', '#DCE3EC')]] + [plt.Line2D([], [], marker='*', ls='', color='white', markeredgecolor='black', markersize=16, label='mean per cancer type and sex')]
        axes[0].legend(handles=cancer_legend, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=8, frameon=False, fontsize=13, handletextpad=0.1, columnspacing=0.9)
        axes[1].legend(handles=sex_legend, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, fontsize=13, handletextpad=0.1, columnspacing=1.2)
        fig.tight_layout()
        fig.savefig(output_dir / f'{filename}.png', dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        plot_metadata[key] = {t: {'run': r.run_name, 'd': int(r.latent_dim), 'seed': int(r.seed), 'bd': format_beta_per_dim(float(r.beta_d)), 'auc': float(r['auc_cancer' if t == 'cancer' else 'auc_sex'])} for t, r in selected_runs[key].items()}
        class_counts = pd.crosstab(clinical_metadata.CANCER_TYPE_ACRONYM.astype(str), clinical_metadata.SEX.astype(str))
        plot_metadata['counts'] = {k: {s: int(class_counts.loc[k, s]) for s in class_counts.columns} for k in class_counts.index}
        json.dump({**{key: value for key, value in plot_metadata.items() if key != 'counts'}, 'counts': plot_metadata['counts']}, open(output_dir / 'latent2d.json', 'w'), ensure_ascii=False, indent=1)
    return draw_latent_view

draw_latent_view = prepare_latent_views(data, OUTPUT)


In [ ]:
draw_latent_view('best', '16_latent2d_best')
show_figures('16_latent2d_best.png')


## Folie 11 — 16c_latent2d_best_cancer_model.png


In [ ]:
draw_latent_view('best_cmodel', '16c_latent2d_best_cancer_model')
show_figures('16c_latent2d_best_cancer_model.png')


## Folie 12 — 16d_latent2d_best_sex_model.png


In [ ]:
draw_latent_view('best_smodel', '16d_latent2d_best_sex_model')
show_figures('16d_latent2d_best_sex_model.png')


## Folie 13 — 16b_latent2d_split.png


In [ ]:
draw_latent_view('split', '16b_latent2d_split')
show_figures('16b_latent2d_split.png')


## Folie 14 — 17_latent2d_collapse.png


In [ ]:
draw_latent_view('collapse', '17_latent2d_collapse')
show_figures('17_latent2d_collapse.png')


## Folie 15 — 17b_latent2d_collapse_cancer_model.png


In [ ]:
draw_latent_view('coll_cmodel', '17b_latent2d_collapse_cancer_model')
show_figures('17b_latent2d_collapse_cancer_model.png')


## Folie 16 — 17c_latent2d_collapse_sex_model.png


In [ ]:
draw_latent_view('coll_smodel', '17c_latent2d_collapse_sex_model')
show_figures('17c_latent2d_collapse_sex_model.png')


## Zusätzliche Latenträume — alle acht TCGA-Aufgaben

Für denselben Beispielsweep (d = 8, Seed 13) zeigen wir β/d = 0, 5·10⁻⁵ und 10⁻³. Alle Aufgaben nutzen dieselben gespeicherten z-UMAP-Koordinaten. Graue Punkte liegen außerhalb der Aufgabe oder gehören zu ausgeschlossenen Klassen; Sterne sind Klassenmittelpunkte im 2D-Plot. Die angegebenen Validierungs-AUCs stammen aus der nativen AUTOENCODIX-Probe auf einem separaten z-Sample im vollständigen Latentraum.


In [ ]:
def plot_additional_latent_spaces(data, output_dir, latent_dim=8, seed=13,
                                 beta_per_dim=(0., 5e-5, 1e-3), selected_task=None):
    """Show all eight TCGA task colourings on the SAME three cached embeddings.

    Class eligibility matches the native probes (>=20 training samples).
    Stars are native 2D class centres; sex centres are grouped by cancer and sex.
    Validation AUCs concern a separate full-dimensional z draw, not the UMAP.
    """
    import io
    from textwrap import fill
    from matplotlib.lines import Line2D

    titles = {
        'cancer_type': 'Cancer type', 'cancer_groups': 'Cancer groups',
        'brca_subtypes': 'BRCA molecular subtype', 'ucec_subtypes': 'UCEC molecular subtype',
        'detailed_histology': 'Detailed histology', 'stage_I_IV': 'Tumour stage (I–IV)',
        'sex_all': 'Sex · all cancer types', 'sex_mixed': 'Sex · mixed cancer types',
    }
    clinical = data['clinical_metadata']
    scores = pd.read_parquet(output_dir / 'downstream_all_tasks.parquet')
    examples = []
    for beta in beta_per_dim:
        candidates = data['run_metrics'][(data['run_metrics'].latent_dim == latent_dim)
                                        & (data['run_metrics'].seed == seed)
                                        & np.isclose(data['run_metrics'].beta_d, beta, rtol=0, atol=1e-10)]
        if len(candidates) != 1:
            raise ValueError(f'Expected one model: d={latent_dim}, seed={seed}, beta/d={beta}')
        run = candidates.iloc[0]
        embedding = pd.read_csv(data['sweep_dir'] / 'plots' / run.run_name / '2D_embedding_z_all.csv', index_col=0)
        embedding.index = embedding.index.astype(str)
        train_ids = data['load_posterior'](run.run_name, 'train')[2]
        expected = np.concatenate([data['load_posterior'](run.run_name, split)[2]
                                   for split in ['train', 'valid', 'test']])
        if not embedding.index.is_unique or len(embedding) != len(expected) or set(embedding.index) != set(expected):
            raise ValueError('UMAP/posterior sample-ID mismatch')
        embedding = embedding.reindex(expected)
        if embedding.shape[1] != 2 or not np.isfinite(embedding.to_numpy()).all():
            raise ValueError('Invalid cached UMAP coordinates')
        examples.append((run, embedding, train_ids))

    audit, filenames = [], []
    for task, (target, cohort) in TASKS.items():
        if selected_task is not None and task != selected_task:
            continue
        figure, axes = plt.subplots(1, len(examples), figsize=(9 * len(examples), 8.5))
        axes = np.atleast_1d(axes)
        for axis, (run, embedding, train_ids) in zip(axes, examples):
            annotations = clinical.reindex(embedding.index)
            raw_labels = annotations[target].astype('string').str.strip()
            train_annotations = clinical.reindex(train_ids)
            train_labels = train_annotations[target].astype('string').str.strip()
            eligible_train = train_labels.notna() & ~train_labels.str.lower().isin(['unknown', '', 'nan', 'none'])
            if cohort:
                eligible_train &= train_annotations.CANCER_TYPE_ACRONYM.isin(cohort)
            counts = train_labels[eligible_train].value_counts()
            classes = sorted(counts[counts >= 20].index.astype(str))
            inside = annotations.CANCER_TYPE_ACRONYM.isin(cohort) if cohort else pd.Series(True, index=annotations.index)
            unknown = raw_labels.isna() | raw_labels.str.lower().isin(['unknown', '', 'nan', 'none'])
            labels = raw_labels.where(inside & raw_labels.isin(classes), 'other')
            labels = labels.mask(inside & unknown, 'unknown').astype(str)
            palette = {label: plt.get_cmap('tab20' if len(classes) > 10 else 'tab10')(i)
                       for i, label in enumerate(classes)}
            if target == 'SEX':
                palette.update(Female='#00A6A6', Male='#1A1A1A')
                native_labels = [f'{cancer} / {label}' if label in classes else label
                                 for cancer, label in zip(annotations.CANCER_TYPE_ACRONYM, labels)]
                native_palette = {compound: palette[label]
                                  for compound, label in zip(native_labels, labels) if label in classes}
            else:
                native_labels, native_palette = labels.tolist(), palette.copy()
            palette.update(other='#DCE3EC', unknown='#9AA6B5')
            native_palette.update(other='#DCE3EC', unknown='#9AA6B5')
            auc = scores[(scores.run_name == run.run_name) & (scores.task == task)
                         & (scores.score_split == 'valid')].value
            if len(auc) != 1:
                raise ValueError(f'Missing/ambiguous validation score: {task}, {run.run_name}')
            title = f'β/d = {data["format_beta"](run.beta_d)} · valid AUC {auc.iloc[0]:.3f}'
            native = _native_latent_panel(embedding.reset_index(drop=True), native_labels, native_palette,
                                         title, '#00A6A6' if target == 'SEX' else '#2F6DB5')
            with io.BytesIO() as buffer:
                native.savefig(buffer, format='png', dpi=170, bbox_inches='tight', facecolor='white')
                buffer.seek(0)
                axis.imshow(plt.imread(buffer))
            plt.close(native)
            axis.axis('off')
            audit.append(dict(task=task, run_name=run.run_name, beta_per_dim=run.beta_d,
                              latent_dim=latent_dim, seed=seed, valid_auc=auc.iloc[0],
                              n_points=len(embedding), n_task_points=int(labels.isin(classes).sum()),
                              class_counts=json.dumps(labels.value_counts().to_dict()),
                              classes=json.dumps(classes), representation='cached_z_umap',
                              auc_representation='z_redrawn_seed31'))
        handles = [Line2D([], [], marker='o', linestyle='', color=palette[label], label=fill(label, width=32))
                   for label in classes]
        handles += [Line2D([], [], marker='o', linestyle='', color=palette[label], label=text)
                    for label, text in [('other', 'outside task / excluded class'), ('unknown', 'unknown label')]]
        handles.append(Line2D([], [], marker='*', linestyle='', markerfacecolor='white',
                              markeredgecolor='black', markersize=14,
                              label='2D mean per cancer type and sex' if target == 'SEX' else '2D class mean'))
        figure.suptitle(f'{titles[task]} · d = {latent_dim} · seed = {seed}', fontsize=22)
        figure.legend(handles=handles, loc='lower center', bbox_to_anchor=(.5, .055),
                      ncol=min(5 if len(classes) > 10 else 6, len(handles)), frameon=False, fontsize=10)
        figure.text(.5, .012, 'Same cached z-UMAP for every task · all data splits shown · '
                    'AUC evaluated on validation in full latent space using a separate z draw (seed 31)',
                    ha='center', fontsize=12)
        figure.tight_layout(rect=(0, .25 if len(classes) > 10 else .17, 1, .95))
        filename = f'latent_tasks_{task}_d{latent_dim}_seed{seed}.png'
        figure.savefig(output_dir / filename, dpi=170, bbox_inches='tight', facecolor='white')
        plt.close(figure)
        filenames.append(filename)
    audit_path = output_dir / 'additional_latent_space_examples.parquet'
    audit_frame = pd.DataFrame(audit)
    if audit_path.exists():
        audit_frame = pd.concat([pd.read_parquet(audit_path), audit_frame], ignore_index=True)
        audit_frame = audit_frame.drop_duplicates(['task', 'run_name'], keep='last')
    audit_frame.to_parquet(audit_path, index=False)
    return filenames


In [ ]:
latent_task_files = []


### Krebstyp


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='cancer_type')
show_figures('latent_tasks_cancer_type_d8_seed13.png')


### Krebsgruppen


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='cancer_groups')
show_figures('latent_tasks_cancer_groups_d8_seed13.png')


### BRCA-Subtypen


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='brca_subtypes')
show_figures('latent_tasks_brca_subtypes_d8_seed13.png')


### UCEC-Subtypen


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='ucec_subtypes')
show_figures('latent_tasks_ucec_subtypes_d8_seed13.png')


### Detaillierte Histologie


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='detailed_histology')
show_figures('latent_tasks_detailed_histology_d8_seed13.png')


### Tumorstadium


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='stage_I_IV')
show_figures('latent_tasks_stage_I_IV_d8_seed13.png')


### Geschlecht — alle Krebsarten


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='sex_all')
show_figures('latent_tasks_sex_all_d8_seed13.png')


### Geschlecht — gemischte Krebsarten


In [ ]:
latent_task_files += plot_additional_latent_spaces(data, OUTPUT, selected_task='sex_mixed')
show_figures('latent_tasks_sex_mixed_d8_seed13.png')


## Klassenstruktur und Downstream-AUC

Silhouette und kNN-Klassenreinheit werden für alle acht Aufgaben auf denselben Validierungsproben und demselben z-Sample wie die native AUC berechnet. Keine UMAP-Distanzen und kein Zusammenlegen von Validierung und Test. Die Klassenauswahl erfolgt wie bei der Probe anhand der Trainingsdaten. Die zufällig erwartete Klassenreinheit berücksichtigt die Klassengrößen.


In [ ]:
TITLES = {
    'cancer_type': 'Cancer type', 'cancer_groups': 'Cancer groups',
    'brca_subtypes': 'BRCA molecular subtype', 'ucec_subtypes': 'UCEC molecular subtype',
    'detailed_histology': 'Detailed histology', 'stage_I_IV': 'Tumour stage (I–IV)',
    'sex_all': 'Sex · all cancer types', 'sex_mixed': 'Sex · mixed cancer types',
}

def compute_task_geometry(data, output_dir):
    """Evaluate the VALIDATION cohort and exact z draw used for its AUC.

    No UMAP or pooled valid/test data. Class eligibility is decided on train.
    Neighbour queries omit each point itself. Purity's chance level accounts
    for the observed class sizes; silhouette uses sklearn's Euclidean metric.
    """
    output_dir = Path(output_dir)
    path = output_dir / 'downstream_task_geometry.parquet'
    aucs = pd.read_parquet(output_dir / 'downstream_all_tasks.parquet')
    aucs = aucs[aucs.score_split.eq('valid')]
    runs = data['run_metrics']
    if path.exists():
        cached = pd.read_parquet(path)
        if set(cached.run_name) == set(runs.run_name) and len(cached) == len(runs) * len(TASKS):
            merged = cached.drop(columns='auc').merge(
                aucs[['run_name', 'task', 'value']].rename(columns={'value': 'auc'}),
                on=['run_name', 'task'], validate='one_to_one')
            return merged
    rows = []
    clinical = data['clinical_metadata']
    for number, run in enumerate(runs.itertuples(), 1):
        frames = load_latent_frames(data['sweep_dir'], clinical, run.run_name)
        for task, (target, cohort) in TASKS.items():
            selected, classes = select_task_frames(frames, clinical, target, cohort)
            frame = selected['valid']
            labels = clinical.loc[frame.index, target].to_numpy()
            counts = pd.Series(labels).value_counts()
            n = len(frame)
            valid = 2 <= len(counts) < n and set(counts.index) == classes
            silhouette, purity, adjusted, chance = (np.nan,) * 4
            k = min(20, n - 1)
            if valid:
                silhouette = silhouette_score(frame.to_numpy(), labels, metric='euclidean')
                neighbours = NearestNeighbors(n_neighbors=k).fit(frame).kneighbors(return_distance=False)
                purity = float((labels[neighbours] == labels[:, None]).mean())
                chance = float((counts * (counts - 1)).sum() / (n * (n - 1)))
                adjusted = (purity - chance) / (1 - chance)
            score = aucs[(aucs.run_name == run.run_name) & aucs.task.eq(task)]
            if len(score) != 1:
                raise ValueError(f'Missing/ambiguous AUC: {task}, {run.run_name}')
            rows.append(dict(run_name=run.run_name, task=task, latent_dim=run.latent_dim,
                             seed=run.seed, beta_per_dim=run.beta_d, auc=score.value.iloc[0],
                             silhouette=silhouette, purity=purity, purity_chance=chance,
                             purity_adjusted=adjusted, n_samples=n, n_classes=len(counts), k=k,
                             split='valid', representation='z_redrawn', random_state=31,
                             status='ok' if valid else 'missing_or_invalid_evaluation_classes'))
        if number % 40 == 0:
            print(f'Label-based geometry: {number}/{len(runs)} models')
    result = pd.DataFrame(rows)
    result.to_parquet(path, index=False)
    return result

def trajectory_correlations(geometry, output_dir):
    """Descriptive value correlations across beta within each trajectory."""
    rows = []
    for (task, dimension, seed), group in geometry.groupby(['task', 'latent_dim', 'seed']):
        for metric in ['silhouette', 'purity', 'purity_adjusted']:
            pairs = group[[metric, 'auc']].dropna()
            enough = len(pairs) >= 4 and pairs.nunique().min() > 1
            rows.append(dict(task=task, latent_dim=dimension, seed=seed, metric=metric,
                             n=len(pairs), pearson=pearsonr(pairs[metric], pairs.auc).statistic if enough else np.nan,
                             spearman=spearmanr(pairs[metric], pairs.auc).statistic if enough else np.nan))
    result = pd.DataFrame(rows)
    result.to_parquet(Path(output_dir) / 'task_geometry_auc_correlations.parquet', index=False)
    return result


In [ ]:
geometry = compute_task_geometry(data, OUTPUT)
geometry_correlations = trajectory_correlations(geometry, OUTPUT)


### Silhouette im Verhältnis zur AUC

Jede Linie gehört zu einer Latentdimension; jeder Punkt ist der Median über vier Seeds bei festem β/d. Die Sterne markieren β = 0.


In [ ]:
def plot_geometry_auc_relationship(geometry, output_dir, selected_metric=None):
    """Connect beta-wise medians of FOUR seeds separately for each dimension."""
    summary = geometry.groupby(['task', 'latent_dim', 'beta_per_dim']).median(numeric_only=True).reset_index()
    dimensions = sorted(geometry.latent_dim.unique())
    filenames = []
    for metric, label in [('silhouette', 'Silhouette'), ('purity_adjusted', 'Chance-adjusted kNN class purity')]:
        if selected_metric is not None and metric != selected_metric:
            continue
        figure, axes = plt.subplots(2, 4, figsize=(21, 10), sharey=True)
        for axis, (task, title) in zip(axes.flat, TITLES.items()):
            for index, dimension in enumerate(dimensions):
                values = summary[summary.task.eq(task) & summary.latent_dim.eq(dimension)].sort_values('beta_per_dim')
                axis.plot(values[metric], values.auc, marker='o', markersize=4, alpha=.8,
                          color=plt.get_cmap('tab10')(index), label=f'd = {dimension}')
                axis.scatter(values[metric].iloc[0], values.auc.iloc[0], marker='*', s=130,
                             color=plt.get_cmap('tab10')(index), edgecolor='black', linewidth=.5, zorder=5)
            axis.set_title(title)
            axis.set_xlabel(label)
            axis.axhline(.5, color='#9AA6B5', linestyle=':', linewidth=1)
            axis.set_ylim(.35, 1.02)
            axis.grid(alpha=.2)
        for axis in axes[:, 0]:
            axis.set_ylabel('Validation macro-OvO AUC')
        handles, labels = axes.flat[0].get_legend_handles_labels()
        figure.legend(handles, labels, loc='lower center', ncol=len(dimensions), frameon=False)
        figure.suptitle(f'{label} versus downstream AUC', fontsize=19)
        figure.text(.5, .045, 'Each point = median across four seeds at fixed d and β/d · '
                    'lines follow increasing β · star = β = 0 · correlation does not establish preservation',
                    ha='center', fontsize=11)
        figure.tight_layout(rect=(0, .08, 1, .94))
        filename = f'{metric}_vs_downstream_auc.png'
        figure.savefig(Path(output_dir) / filename, dpi=170, bbox_inches='tight')
        plt.close(figure)
        filenames.append(filename)
    return filenames


In [ ]:
plot_geometry_auc_relationship(geometry, OUTPUT, selected_metric='silhouette')
show_figures('silhouette_vs_downstream_auc.png')


### Klassenreinheit im Verhältnis zur AUC

Für den Vergleich der Aufgaben wird die Klassenreinheit um ihre zufällig erwartete Überlappung korrigiert.


In [ ]:
plot_geometry_auc_relationship(geometry, OUTPUT, selected_metric='purity_adjusted')
show_figures('purity_adjusted_vs_downstream_auc.png')


### Pearson und Spearman je Trajektorie

Zuerst Korrelation über die β-Werte einer Trajektorie; danach Median über vier Seeds je Aufgabe und Latentdimension. Beschreibende Zusammenhänge, keine Kausalität und kein unabhängiger Nachweis eines Frühwarnsignals.


In [ ]:
display(geometry_correlations.groupby(['task', 'latent_dim', 'metric'])[['pearson', 'spearman']].median().round(3))


### β-Verlauf: Krebstyp

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
def plot_task_geometry(geometry, data, output_dir, latent_dim=8, tasks=None):
    """For each task, plot AUC and geometry along beta, median/IQR of 4 seeds."""
    filenames = []
    for task, title in TITLES.items():
        if tasks is not None and task not in tasks:
            continue
        selected = geometry[geometry.task.eq(task) & geometry.latent_dim.eq(latent_dim)]
        grouped = selected.groupby('beta_per_dim')
        median = grouped.median(numeric_only=True).reindex(data['beta_grid'])
        lower = grouped.quantile(.25, numeric_only=True).reindex(data['beta_grid'])
        upper = grouped.quantile(.75, numeric_only=True).reindex(data['beta_grid'])
        figure, axes = plt.subplots(1, 3, figsize=(19, 5.8), sharex=True)
        panels = [('auc', 'Validation macro-OvO AUC', '#1B2430'),
                  ('silhouette', 'Silhouette · Euclidean', '#2F6DB5'),
                  ('purity', 'kNN class purity · k = 20', '#D9822B')]
        for axis, (metric, label, color) in zip(axes, panels):
            axis.plot(data['beta_positions'], median[metric], marker='o', color=color, linewidth=2.5)
            axis.fill_between(data['beta_positions'], lower[metric], upper[metric], color=color, alpha=.15)
            axis.set_ylabel(label)
            data['format_beta_axis'](axis)
            axis.grid(alpha=.2)
        axes[0].axhline(.5, color='#9AA6B5', linestyle=':')
        axes[0].set_ylim(.4, 1.02)
        axes[1].axhline(0, color='#9AA6B5', linestyle=':')
        axes[2].plot(data['beta_positions'], median.purity_chance, linestyle=':', color='#9AA6B5',
                     label='chance level from class sizes')
        axes[2].legend(frameon=False, fontsize=10)
        axes[2].set_ylim(0, 1.02)
        figure.suptitle(f'{title} · d = {latent_dim} · median and IQR across four seeds', fontsize=17)
        figure.text(.5, .01, 'Same validation samples and full-dimensional z draw as AUC · '
                    'training defines eligible classes · descriptive comparison, not a causal test',
                    ha='center', fontsize=11)
        figure.tight_layout(rect=(0, .05, 1, .93))
        filename = f'task_geometry_{task}_d{latent_dim}.png'
        figure.savefig(Path(output_dir) / filename, dpi=170, bbox_inches='tight')
        plt.close(figure)
        filenames.append(filename)
    return filenames


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['cancer_type'])
show_figures('task_geometry_cancer_type_d8.png')


### β-Verlauf: Krebsgruppen

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['cancer_groups'])
show_figures('task_geometry_cancer_groups_d8.png')


### β-Verlauf: BRCA-Subtypen

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['brca_subtypes'])
show_figures('task_geometry_brca_subtypes_d8.png')


### β-Verlauf: UCEC-Subtypen

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['ucec_subtypes'])
show_figures('task_geometry_ucec_subtypes_d8.png')


### β-Verlauf: Histologie

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['detailed_histology'])
show_figures('task_geometry_detailed_histology_d8.png')


### β-Verlauf: Tumorstadium

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['stage_I_IV'])
show_figures('task_geometry_stage_I_IV_d8.png')


### β-Verlauf: Geschlecht — alle Krebsarten

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['sex_all'])
show_figures('task_geometry_sex_all_d8.png')


### β-Verlauf: Geschlecht — gemischte Krebsarten

AUC, Silhouette und Klassenreinheit bei d = 8: Median und Interquartilsabstand über vier Seeds.


In [ ]:
plot_task_geometry(geometry, data, OUTPUT, latent_dim=8, tasks=['sex_mixed'])
show_figures('task_geometry_sex_mixed_d8.png')


## Welche Metriken messen Ähnliches?

Spearman-Korrelationen werden zwischen Metriken innerhalb jeder Trajektorie berechnet und anschließend als Median der absoluten Korrelationen über Trajektorien dargestellt. Es werden Werte, Änderungen pro log-β-Schritt und partielle Rangkorrelationen unter Kontrolle der Rate verglichen. Bei perfekter Rangkorrelation mit der Rate ist die partielle Korrelation undefiniert (grau). Hohe Ähnlichkeit ist ein Hinweis auf Redundanz, keine Gleichwertigkeit und keine Aussage zur AUC. Die untersuchten Zustände entsprechen dem bestehenden Bereich vor dem starken Kollaps.


In [ ]:
def metric_family(name):
    """Group by measurement target; graph/topology may span multiple scales."""
    if name.startswith('geometry_'):
        if any(term in name for term in ['steadiness', 'cohesiveness']):
            return 'Groups'
        if any(term in name for term in ['trustworthiness', 'continuity', 'lcmc', 'mrre', 'neighbor_dissimilarity']):
            return 'Local neighbourhoods'
        if 'kl_divergence' in name:
            return 'Density'
        return 'Global geometry / other geometry'
    if name.startswith('dtm_'):
        return 'Density'
    if name.startswith('topology_'):
        return 'Topology (multiple scales)'
    if name.startswith('dimensionality_'):
        return 'Dimensionality'
    if name.startswith('info_'):
        return 'Information'
    if name.startswith('graph_'):
        return 'Graph structure (multiple scales)'
    return 'Other'

def compute_redundancy(data, output_dir):
    """Compare values, log-beta slopes, and rank correlations controlling log-rate.

    No pooling of adjacent beta points across trajectories. Opposite directions
    count as similar via absolute rho; signed medians and sign consistency are
    also saved. Partial rank correlations are NaN for perfectly rate-monotone
    metrics; they are not independent validation or estimates of causality.
    """
    metrics = sorted(data['usable_metrics'], key=lambda name: (metric_family(name), name))
    rows = []
    for (dimension, seed), group in data['admissible_states'].groupby(['latent_dim', 'seed']):
        group = group.sort_values('beta')
        values = group[metrics]
        raw = values.corr(method='spearman', min_periods=4)
        slopes = values.diff().iloc[1:].div(np.diff(np.log(group.beta.to_numpy())), axis=0)
        changes = slopes.corr(method='spearman', min_periods=4)
        with_rate = group[metrics + ['logR']].corr(method='spearman', min_periods=4)
        rate_rho = with_rate.loc[metrics, 'logR'].to_numpy()
        denominator = np.sqrt(np.maximum(0, 1 - rate_rho**2))
        denominator = denominator[:, None] * denominator[None, :]
        partial = np.divide(raw.to_numpy() - np.outer(rate_rho, rate_rho), denominator,
                            out=np.full(raw.shape, np.nan), where=denominator > 1e-10)
        partial = pd.DataFrame(np.clip(partial, -1, 1), index=metrics, columns=metrics)
        triangle = np.triu(np.ones(raw.shape, dtype=bool), k=1)
        for method, matrix, n in [('values', raw, len(values)), ('slopes', changes, len(slopes)),
                                  ('partial_rank_rate', partial, len(values))]:
            pairs = matrix.where(triangle).stack().dropna().rename('rho').reset_index()
            pairs.columns = ['metric_a', 'metric_b', 'rho']
            rows.append(pairs.assign(method=method, latent_dim=dimension, seed=seed, n_steps=n))
    trajectories = pd.concat(rows, ignore_index=True)
    trajectories['abs_rho'] = trajectories.rho.abs()
    summary = trajectories.groupby(['method', 'metric_a', 'metric_b']).agg(
        median_abs_rho=('abs_rho', 'median'), median_signed_rho=('rho', 'median'),
        q25_abs_rho=('abs_rho', lambda values: values.quantile(.25)),
        n_trajectories=('rho', 'count'), positive_fraction=('rho', lambda values: values.gt(0).mean()),
        min_steps=('n_steps', 'min'), max_steps=('n_steps', 'max')).reset_index()
    summary['family_a'] = summary.metric_a.map(metric_family)
    summary['family_b'] = summary.metric_b.map(metric_family)
    trajectories.to_parquet(Path(output_dir) / 'metric_redundancy_by_trajectory.parquet', index=False)
    summary.to_parquet(Path(output_dir) / 'metric_redundancy_summary.parquet', index=False)
    return metrics, summary


In [ ]:
def correlation_matrix(metrics, summary, method):
    values = np.full((len(metrics), len(metrics)), np.nan)
    np.fill_diagonal(values, 1.)
    matrix = pd.DataFrame(values, index=metrics, columns=metrics)
    selected = summary[summary.method.eq(method) & summary.metric_a.isin(metrics) & summary.metric_b.isin(metrics)]
    for pair in selected.itertuples():
        matrix.loc[pair.metric_a, pair.metric_b] = pair.median_abs_rho
        matrix.loc[pair.metric_b, pair.metric_a] = pair.median_abs_rho
    return matrix

def save_redundancy_groups(metrics, summary, data, output_dir):
    changes = correlation_matrix(metrics, summary, 'slopes').fillna(0).to_numpy()
    distances = np.clip(1 - changes, 0, 1)
    np.fill_diagonal(distances, 0)
    tree = linkage(squareform(distances, checks=True), method='complete')
    from scipy.cluster.hierarchy import fcluster
    groups = fcluster(tree, t=0.1, criterion='distance')
    pd.DataFrame({'metric': metrics, 'family': [metric_family(name) for name in metrics], 'exploratory_group_at_rho_0p90': groups}).to_csv(Path(output_dir) / 'metric_redundancy_groups.csv', index=False)


In [ ]:
redundancy_metrics, redundancy_summary = compute_redundancy(data, OUTPUT)
save_redundancy_groups(redundancy_metrics, redundancy_summary, data, OUTPUT)


### Geometrie-Metriken: Ähnliche Werte entlang β


In [ ]:
def plot_redundancy(metrics, summary, data, output_dir, subset_name=None, method_name=None):
    """Heatmaps for all metrics and a larger-text geometry subset."""
    filenames = []
    subsets = [('all', metrics), ('geometry', [name for name in metrics if name.startswith('geometry_')])]
    for subset, selected in subsets:
        if subset_name is not None and subset != subset_name:
            continue
        if len(selected) < 2:
            continue
        for method, description in [('values', 'metric values'), ('slopes', 'changes per log β step'), ('partial_rank_rate', 'partial ranks controlling log rate')]:
            if method_name is not None and method != method_name:
                continue
            matrix = correlation_matrix(selected, summary, method)
            size = max(9, len(selected) * 0.32)
            figure, axis = plt.subplots(figsize=(size + 3, size))
            palette = plt.get_cmap('viridis').copy()
            palette.set_bad('#DCE3EC')
            image = axis.imshow(matrix, vmin=0, vmax=1, cmap=palette)
            labels = [data['metric_label'](name) for name in selected]
            axis.set_xticks(range(len(selected)), labels, rotation=90, fontsize=8)
            axis.set_yticks(range(len(selected)), labels, fontsize=8)
            families = [metric_family(name) for name in selected]
            for position in range(1, len(families)):
                if families[position] != families[position - 1]:
                    axis.axhline(position - 0.5, color='white', linewidth=1)
                    axis.axvline(position - 0.5, color='white', linewidth=1)
            axis.set_title(f'Metric redundancy · {description}\nMedian |Spearman ρ| within trajectories', fontsize=14)
            figure.colorbar(image, ax=axis, fraction=0.03, pad=0.02)
            figure.text(0.5, 0.01, 'Grey = undefined · positive β, pre-collapse analysis states · high correlation suggests redundancy; it does not establish equivalence', ha='center', fontsize=10)
            figure.tight_layout(rect=(0, 0.035, 1, 1))
            filename = f'metric_redundancy_{subset}_{method}.png'
            figure.savefig(Path(output_dir) / filename, dpi=170, bbox_inches='tight')
            plt.close(figure)
            filenames.append(filename)
    return filenames


In [ ]:
plot_redundancy(redundancy_metrics, redundancy_summary, data, OUTPUT, subset_name='geometry', method_name='values')
show_figures('metric_redundancy_geometry_values.png')


### Geometrie-Metriken: Ähnliche Veränderungen entlang β


In [ ]:
plot_redundancy(redundancy_metrics, redundancy_summary, data, OUTPUT, subset_name='geometry', method_name='slopes')
show_figures('metric_redundancy_geometry_slopes.png')


### Geometrie-Metriken: Zusammenhang nach Kontrolle der Rate


In [ ]:
plot_redundancy(redundancy_metrics, redundancy_summary, data, OUTPUT, subset_name='geometry', method_name='partial_rank_rate')
show_figures('metric_redundancy_geometry_partial_rank_rate.png')


### Alle Metrikfamilien: gemeinsame Veränderungen


In [ ]:
plot_redundancy(redundancy_metrics, redundancy_summary, data, OUTPUT, subset_name='all', method_name='slopes')
show_figures('metric_redundancy_all_slopes.png')


### Kandidaten für eine reduzierte Hauptauswertung

Explorative Auswahl: Median |ρ| ≥ 0,90, unteres Quartil ≥ 0,80 und mindestens 20 Trajektorien für Änderungen. Die Schwellen sind gewählte Arbeitskriterien. Vertreter müssen fachlich dieselbe Frage beantworten; es wird nichts automatisch aus der Pipeline entfernt.


In [ ]:
similar_changes = redundancy_summary.query(
    "method == 'slopes' and median_abs_rho >= 0.90 and q25_abs_rho >= 0.80 and n_trajectories >= 20"
).sort_values('median_abs_rho', ascending=False)
display(similar_changes[['metric_a', 'metric_b', 'family_a', 'family_b',
                         'median_abs_rho', 'q25_abs_rho', 'n_trajectories']])


## Folie 21 — 38_metric_heatmap.png


In [ ]:
def plot_metric_heatmap(data, output_dir):
    """Original raw and same-beta correlation heatmaps."""
    admissible_states = data['admissible_states']
    hidden_metrics = data['hidden_metrics']
    metric_label = data['metric_label']
    usable_metrics = data['usable_metrics']
    import seaborn as sns
    from scipy.cluster.hierarchy import linkage, leaves_list
    from scipy.spatial.distance import squareform
    from matplotlib.patches import Patch
    correlation_columns = ['logR', 'auc_cancer', 'auc_sex'] + usable_metrics

    def _res(g, c):
        X = np.column_stack([np.ones(len(g)), g.logR, g.logR ** 2])
        y = g[c].to_numpy(float)
        return y - X @ np.linalg.lstsq(X, y, rcond=None)[0]

    def mean_corr12(cols, adjust):
        zs = []
        for _, g in admissible_states.groupby(['latent_dim', 'seed']):
            M = pd.DataFrame({c: _res(g, c) if adjust else g[c].to_numpy(float) for c in cols})
            zs.append(np.arctanh(np.clip(M.corr(method='spearman').to_numpy(), -0.999999,
                0.999999)))
        a = np.tanh(np.nanmean(zs, axis=0))
        np.fill_diagonal(a, 1.0)
        return pd.DataFrame(a, index=cols, columns=cols)
    raw_correlations = mean_corr12(correlation_columns, False)

    def cell_corr12(cols):
        zs = []
        for _, g in admissible_states.groupby(['latent_dim', 'beta_d']):
            M = g[cols].astype(float)
            ok = [c for c in cols if M[c].nunique() >= 3]
            R = M.corr(method='spearman').where(pd.DataFrame(True, index=cols, columns=cols).loc[ok,
                ok].reindex(index=cols, columns=cols).fillna(False))
            zs.append(np.arctanh(np.clip(R.to_numpy(float), -0.999999, 0.999999)))
        a = np.tanh(np.nanmean(zs, axis=0))
        np.fill_diagonal(a, 1.0)
        return pd.DataFrame(a, index=cols, columns=cols)
    same_beta_correlations = cell_corr12([c for c in correlation_columns if c != 'logR'])
    Dm = np.nan_to_num(1 - raw_correlations.loc[usable_metrics, usable_metrics].abs().to_numpy(),
        nan=1.0)
    np.fill_diagonal(Dm, 0)
    Dm = (Dm + Dm.T) / 2
    metric_order = ['logR', 'auc_cancer',
        'auc_sex'] + [usable_metrics[i] for i in leaves_list(linkage(squareform(Dm, checks=False),
        method='average'))]

    def fam12(c):
        if c == 'logR':
            return 'Rate'
        if c.startswith('auc'):
            return 'AUC'
        if 'forman' in c:
            return 'Curvature'
        if c in ('graph_partition_ari_k20', 'graph_partition_nmi_k20',
            'graph_hub_identity_spearman_k20', 'graph_directed_edge_jaccard_lcmc_related_k20'):
            return 'Graph comparison'
        for p, f in [('dimensionality_', 'Dimensionality'), ('topology_', 'TDA'), ('dtm_', 'TDA'),
            ('geometry_', 'Geometry'), ('graph_', 'Graph'), ('info_', 'Information')]:
            if c.startswith(p):
                return f
        return 'Other'
    FAMC = {'Rate': '#1B2430', 'AUC': '#C0392B', 'Geometry': '#2F6DB5',
        'Graph comparison': '#8C564B', 'TDA': '#00A6A6', 'Graph': '#2E8B57',
        'Dimensionality': '#D9822B', 'Curvature': '#9467BD', 'Information': '#7F7F7F',
        'Other': '#BBBBBB'}
    metric_order = [c for c in metric_order if c not in hidden_metrics]
    metric_labels = [{'logR': 'RATE (log R)', 'auc_cancer': 'AUC cancer type',
        'auc_sex': 'AUC sex (mixed)'}.get(c, metric_label(c)) for c in metric_order]
    iu = np.triu_indices(len(usable_metrics), 1)
    for name, C in [('über β', raw_correlations), ('gleiches β (über Seeds)',
        same_beta_correlations)]:
        v = np.abs(C.loc[usable_metrics, usable_metrics].to_numpy()[iu])
    with matplotlib.rc_context({'font.size': 9}):
        fig, axs = plt.subplots(1, 2, figsize=(30, 14.5))
        for ax, C, title in [(axs[0], raw_correlations,
            'Across β: almost everything moves together with the rate'), (axs[1],
            same_beta_correlations, 'Same β (across seeds): the block falls apart')]:
            o = [c for c in metric_order if c in C.index]
            lab = [metric_labels[metric_order.index(c)] for c in o]
            sns.heatmap(C.loc[o, o], cmap='vlag', vmin=-1, vmax=1, center=0, square=True, ax=ax,
                xticklabels=lab, yticklabels=lab, cbar_kws={'shrink': 0.6,
                'label': 'Spearman ρ (mean within trajectories)'})
            for t in ax.get_yticklabels() + ax.get_xticklabels():
                t.set_color(FAMC[fam12(o[lab.index(t.get_text())])])
            ax.set_title(title, fontsize=15, loc='left', fontweight='bold')
            ax.tick_params(labelsize=7)
        fig.legend(handles=[Patch(color=c, label=f) for f, c in FAMC.items() if f != 'Other'],
            loc='lower center', ncol=9, frameon=False, fontsize=11, title='label colour = family')
        fig.tight_layout(rect=(0, 0.04, 1, 1))
        fig.savefig(output_dir / '38_metric_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='white')
        plt.show()
    raw_correlations.to_csv(output_dir / '30_09_T12_korrelation_roh.csv')
    same_beta_correlations.to_csv(output_dir / '30_09_T12_korrelation_gleiche_rate.csv')


In [ ]:
plot_metric_heatmap(data, OUTPUT)
show_figures('38_metric_heatmap.png')


## Folie 23 — Pearson und Spearman

Beide Korrelationen vergleichen dieselben Metrik- und AUC-Werte innerhalb jeder Trajektorie. Gezeigt werden vorzeichenbehaftete Mediane. Der frühere Vergleich von Werten und Änderungen bleibt unten als ergänzende Analyse erhalten.


In [ ]:
def plot_value_and_drop_correlations(data, output_dir):
    """Values versus slopes from the saved correlation estimates."""
    BLUET = data['BLUET']
    NAVYT = data['NAVYT']
    analysis_plot_style = data['analysis_plot_style']
    course_correlations = data['course_correlations']
    READ6 = {'geometry_mu_trustworthiness_zadu_k20': 'Trustworthiness',
        'geometry_mu_lcmc_zadu_k20': 'LCMC',
        'geometry_mu_non_metric_stress_zadu': 'Stress (non-metric)',
        'geometry_mu_continuity_zadu_k20': 'Continuity',
        'topology_h0_persistence_entropy_mu': 'H0 persistence entropy',
        'graph_latent_forman_std_k20': 'Graph curvature spread',
        'geometry_mu_cohesiveness_zadu_k20': 'Cohesiveness',
        'graph_latent_groupie_ratio_k20': 'Graph groupie ratio',
        'graph_latent_lambda2_k20': 'Graph connectivity (λ₂)',
        'graph_latent_transitivity_k20': 'Graph transitivity',
        'topology_h1_normalized_wasserstein': 'H1 Wasserstein',
        'graph_latent_global_efficiency_k20': 'Graph global efficiency',
        'geometry_mu_scale_normalized_stress_zadu': 'Stress (scale-norm.)'}

    def pair6(ax, C, title, n=6):
        C = C.assign(A=C.A_norm.abs(), r=C.raw.abs())
        t = C[C.index != 'logR'].sort_values('A', ascending=False).head(n).copy()
        t['name'] = [READ6.get(m, nm_) for m, nm_ in zip(t.index, t.name)]
        rr = C.loc[['logR']].copy()
        rr['name'] = 'Rate (reference)'
        t = pd.concat([t, rr]).sort_values('A', ascending=False).iloc[::-1]
        y = np.arange(len(t))
        h = 0.38
        isr = t.index == 'logR'
        ax.barh(y + h / 2, t.r, h, color=np.where(isr, '#7F8C9A', NAVYT))
        ax.barh(y - h / 2, t.A, h, color=np.where(isr, '#B8C2CC', BLUET))
        for yy, (_, r), rate in zip(y, t.iterrows(), isr):
            ax.text(r.r + 0.01, yy + h / 2, f'{r.r:.2f}', va='center', fontsize=12, color=NAVYT)
            ax.text(r.A + 0.01, yy - h / 2, f'{r.A:.2f}', va='center', fontsize=12,
                color='#5E6977' if rate else BLUET, fontweight='bold')
        ax.set_yticks(y)
        ax.set_yticklabels(t.name, fontsize=14)
        for lab, rate in zip(ax.get_yticklabels(), isr):
            if rate:
                lab.set_fontweight('bold')
                lab.set_color('#5E6977')
        ax.set_xlim(0, 1.1)
        ax.set_ylim(-0.7, len(t) - 0.3)
        ax.set_title(title, loc='left', fontweight='bold')
        ax.set_xlabel('|Spearman ρ| with the AUC (median over 24 trajectories)')
        ax.grid(axis='y', visible=False)
    with matplotlib.rc_context(analysis_plot_style):
        fig, ax = plt.subplots(1, 2, figsize=(17, 6.0))
        pair6(ax[0], course_correlations['cancer'], 'Cancer type')
        pair6(ax[1], course_correlations['sex'], 'Sex (mixed types)')
        from matplotlib.patches import Patch
        fig.legend([Patch(color=NAVYT), Patch(color=BLUET), Patch(color='#7F8C9A'),
            Patch(color='#B8C2CC')], ['values across β', 'slopes per log β step', 'rate: values',
            'rate: slopes'], loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.01),
            fontsize=13)
        fig.tight_layout(rect=(0, 0.07, 1, 1))
        fig.savefig(output_dir / '39f_values_vs_drops.png', dpi=200, bbox_inches='tight',
            facecolor='white')
        plt.close(fig)

def plot_correlation_bars(data, output_dir):
    """Slide 23: Pearson and Spearman on the SAME metric/AUC values.

    Keep the six metrics of the earlier values/slopes plot for comparability.
    Compute within trajectories first; show signed medians, not pooled correlations.
    Save the values/slopes comparison separately as 39f_values_vs_drops.png.
    """
    from scipy.stats import pearsonr
    from matplotlib.patches import Patch

    plot_value_and_drop_correlations(data, output_dir)
    records = []
    selected_metrics = {}
    for task, auc_column in [('cancer', 'auc_cancer'), ('sex', 'auc_sex')]:
        course = data['course_correlations'][task]
        selected_metrics[task] = course.drop(index='logR').A_norm.abs().nlargest(6).index.tolist()
        for metric in selected_metrics[task] + ['logR']:
            for (dimension, seed), trajectory in data['admissible_states'].groupby(['latent_dim', 'seed']):
                pairs = trajectory[[metric, auc_column]].replace([np.inf, -np.inf], np.nan).dropna()
                if len(pairs) < 4 or pairs.nunique().min() < 3:
                    continue
                records.append(dict(task=task, metric=metric, latent_dim=dimension, seed=seed,
                                    n=len(pairs), pearson=pearsonr(pairs[metric], pairs[auc_column]).statistic,
                                    spearman=spearmanr(pairs[metric], pairs[auc_column]).statistic))
    correlations = pd.DataFrame(records)
    correlations.to_csv(output_dir / '39e_pearson_spearman_by_trajectory.csv', index=False)
    summary = correlations.groupby(['task', 'metric'])[['pearson', 'spearman']].median()
    summary.to_csv(output_dir / '39e_pearson_spearman_summary.csv')
    with matplotlib.rc_context(data['analysis_plot_style']):
        figure, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=True)
        for axis, task, title in zip(axes, ['cancer', 'sex'], ['Cancer type', 'Sex (mixed types)']):
            values = summary.loc[task].reindex(selected_metrics[task] + ['logR']).iloc[::-1]
            positions = np.arange(len(values))
            axis.barh(positions + .18, values.pearson, height=.36, color='#1B2430')
            axis.barh(positions - .18, values.spearman, height=.36, color='#2F6DB5')
            for position, (_, row) in zip(positions, values.iterrows()):
                for offset, value, color in [(.18, row.pearson, '#1B2430'), (-.18, row.spearman, '#2F6DB5')]:
                    axis.text(value + (.025 if value >= 0 else -.025), position + offset,
                              f'{value:+.2f}', va='center', ha='left' if value >= 0 else 'right',
                              fontsize=11, color=color)
            labels = ['Rate (log R)' if metric == 'logR' else data['metric_label'](metric)
                      for metric in values.index]
            axis.set_yticks(positions, labels)
            axis.axvline(0, color='#9AA6B5', linewidth=1)
            axis.set_xlim(-1.15, 1.15)
            axis.set_title(title, loc='left', fontweight='bold')
            axis.set_xlabel('Signed correlation with AUC · median across trajectories')
            axis.grid(axis='y', visible=False)
        figure.legend(handles=[Patch(color='#1B2430', label='Pearson r · linear relationship'),
                               Patch(color='#2F6DB5', label='Spearman ρ · rank relationship')],
                      loc='lower center', ncol=2, frameon=False)
        figure.suptitle('Same metric values and AUC values across β', fontsize=18)
        figure.tight_layout(rect=(0, .09, 1, .94))
        figure.savefig(output_dir / '39e_values_vs_drops.png', dpi=200,
                       bbox_inches='tight', facecolor='white')
        plt.close(figure)
    return summary


In [ ]:
plot_correlation_bars(data, OUTPUT)
show_figures('39e_values_vs_drops.png')


### Ergänzend: Werte und Änderungen pro log-β-Schritt


In [ ]:
show_figures('39f_values_vs_drops.png')


## Folie 24 — 47b_example_visual.png


In [ ]:
def plot_correlation_example(data, output_dir):
    """Worked example on one actual trajectory, plus rank table."""
    BLUET = data['BLUET']
    NAVYT = data['NAVYT']
    admissible_states = data['admissible_states']
    analysis_plot_style = data['analysis_plot_style']
    format_beta_label = data['format_beta_label']
    from scipy.stats import rankdata
    example_trajectory = admissible_states[(admissible_states.latent_dim == 8) & (admissible_states.seed == 17)].sort_values('beta').reset_index(drop=True)
    mT, mR, aC = ('geometry_mu_trustworthiness_zadu_k20', 'logR', 'auc_cancer')
    lb = np.log(example_trajectory.beta.to_numpy())

    def sl(v):
        return np.diff(v) / np.diff(lb)
    vals = {k: example_trajectory[c].to_numpy(float) for k, c in [('AUC', aC), ('Trust', mT),
        ('log R', mR)]}
    rk = lambda v: rankdata(-v, method='average')
    S = {k: sl(v) for k, v in vals.items()}
    rkS = lambda v: rankdata(v, method='average')
    res = {k: (spearmanr(vals[k], vals['AUC'])[0], spearmanr(S[k], S['AUC'])[0]) for k in ['Trust',
        'log R']}
    for k, (a, b) in res.items():
        pass
    lev = pd.DataFrame({'β/d': [format_beta_label(b) for b in example_trajectory.beta_d],
        'AUC': vals['AUC'], 'rank': rk(vals['AUC']), 'Trust.': vals['Trust'],
        'rank ': rk(vals['Trust']), 'log R': vals['log R'], 'rank  ': rk(vals['log R'])})
    stp = pd.DataFrame({'step': [f'{format_beta_label(a)}→{format_beta_label(b)}' for a,
        b in zip(example_trajectory.beta_d[:-1], example_trajectory.beta_d[1:])],
        'ΔAUC/Δlogβ': S['AUC'], 'rank': rkS(S['AUC']), 'ΔTrust/Δlogβ': S['Trust'],
        'rank ': rkS(S['Trust']), 'ΔlogR/Δlogβ': S['log R'], 'rank  ': rkS(S['log R'])})

    def tab(ax, df, title, fmts, foot):
        ax.axis('off')
        cells = [[f(v) for f, v in zip(fmts, row)] for row in df.itertuples(index=False)]
        w = [2.2 if j == 0 and df.columns[0] == 'step' else 1.0 for j in range(len(df.columns))]
        w = [x / sum(w) for x in w]
        t = ax.table(cellText=cells, colLabels=list(df.columns), cellLoc='center', colWidths=w,
            bbox=[0, 0.24, 1, 0.7])
        t.auto_set_font_size(False)
        t.set_fontsize(12)
        for (r_, c_), cell in t.get_celld().items():
            cell.set_edgecolor('#D5DAE1')
            if r_ == 0:
                cell.set_facecolor('#E8EDF3')
                cell.set_text_props(fontweight='bold', color=NAVYT)
            elif 'rank' in df.columns[c_]:
                cell.set_facecolor('#F4F7FB')
                cell.set_text_props(color=BLUET, fontweight='bold')
        ax.set_title(title, loc='left', fontweight='bold', fontsize=15, color=NAVYT)
        ax.text(0, 0.19, foot, transform=ax.transAxes, fontsize=13, color=NAVYT, va='top')
    f3 = lambda v: f'{v:.3f}'
    f2 = lambda v: f'{v:.2f}'
    fr = lambda v: f'{v:g}'
    fs = lambda v: str(v)
    fe = lambda v: f'{v:+.4f}'
    with matplotlib.rc_context(analysis_plot_style):
        fig, ax = plt.subplots(1, 2, figsize=(19, 5.2), gridspec_kw={'width_ratios': [1, 1.3]})
        tab(ax[0], lev, '1  Values across β (dark bars)', [fs, f3, fr, f3, fr, f2, fr],
            f"ρ_lev = Spearman(values):  Trustworthiness {res['Trust'][0]:+.2f},  rate {res['log R'][0]:+.2f}\nranks nearly identical: all three fall almost every step (AUC rises once, trust. once)")
        tab(ax[1], stp, '2  Slopes per log-β step (blue bars)', [fs, fe, fr, fe, fr, f2, fr],
            f"ρ_slope = Spearman(slopes):  Trustworthiness {res['Trust'][1]:+.2f},  rate {res['log R'][1]:+.2f}\nrank 1 = steepest drop; does the metric drop hardest where the AUC does?")
        fig.tight_layout()
        fig.savefig(output_dir / '47_example_correlations.png', dpi=200, bbox_inches='tight',
            facecolor='white')
        plt.close(fig)
    TJ7j = {'n': len(example_trajectory), 'trust': res['Trust'], 'rate': res['log R']}
    COLS7 = [('AUC', 'cancer-type AUC', NAVYT), ('Trust', 'trustworthiness', BLUET), ('log R',
        'rate (log R)', '#9AA6B5')]
    xs = np.arange(len(example_trajectory))
    xl = [format_beta_label(b) for b in example_trajectory.beta_d]
    sx = np.arange(len(example_trajectory) - 1)
    sl_lab = [f'{format_beta_label(a)}\n→ {format_beta_label(b)}' for a,
        b in zip(example_trajectory.beta_d[:-1], example_trajectory.beta_d[1:])]
    with matplotlib.rc_context(analysis_plot_style):
        fig, ax = plt.subplots(1, 2, figsize=(18, 6.2), gridspec_kw={'width_ratios': [1, 1.25]})
        for k, l, c in COLS7:
            v = vals[k]
            ax[0].plot(xs, (v - v[-1]) / (v[0] - v[-1]), marker='o', lw=3, ms=8, color=c, label=l)
        ax[0].set_xticks(xs)
        ax[0].set_xticklabels(xl, rotation=45)
        ax[0].set_ylabel('scaled value (1 = first β, 0 = last β)')
        ax[0].set_xlabel('β/d')
        ax[0].set_title('1  Values: all three fall (almost)', loc='left', fontweight='bold')
        ax[0].legend(frameon=False, loc='lower left')
        ax[0].text(0.98, 0.98,
            f"ρ (values) with the AUC\ntrustworthiness {res['Trust'][0]:.2f}\nrate {res['log R'][0]:.2f}",
            transform=ax[0].transAxes, ha='right', va='top', fontsize=14, color=NAVYT,
            bbox=dict(boxstyle='round', fc='#F4F7FB', ec='#D5DAE1'))
        w = 0.27
        for j, (k, l, c) in enumerate(COLS7):
            v = S[k]
            v = v / np.abs(v.min())
            ax[1].bar(sx + (j - 1) * w, -v, w, color=c, label=l)
        ax[1].axhline(0, color=NAVYT, lw=1)
        ax[1].set_xticks(sx)
        ax[1].set_xticklabels(sl_lab, fontsize=11)
        ax[1].set_ylabel('size of the drop\n(largest drop = 1; below 0 = rise)')
        ax[1].set_title('2  Drops per β step: are the big ones at the same place?', loc='left',
            fontweight='bold')
        ax[1].text(0.02, 0.98,
            f"ρ (drops) with the AUC\ntrustworthiness {res['Trust'][1]:.2f}\nrate {res['log R'][1]:.2f}: drops almost equally at every step",
            transform=ax[1].transAxes, ha='left', va='top', fontsize=14, color=NAVYT,
            bbox=dict(boxstyle='round', fc='#F4F7FB', ec='#D5DAE1'))
        ax[1].set_ylim(-0.35, 1.45)
        ax[1].grid(axis='x', visible=False)
        fig.tight_layout()
        fig.savefig(output_dir / '47b_example_visual.png', dpi=200, bbox_inches='tight',
            facecolor='white')
        plt.close(fig)


In [ ]:
plot_correlation_example(data, OUTPUT)
show_figures('47b_example_visual.png')


## Folie 26 — 03b_course_tc_coverage.png


In [ ]:
def plot_preservation_course(data, output_dir):
    BLUEW = data['BLUEW']
    GREENW = data['GREENW']
    GREYW = data['GREYW']
    NAVYW = data['NAVYW']
    ORANGEW = data['ORANGEW']
    beta_labels = data['beta_labels']
    beta_positions = data['beta_positions']
    first_lasting_drop = data['first_lasting_drop']
    format_beta_axis = data['format_beta_axis']
    plot_style = data['plot_style']
    plot_summary = data['plot_summary']
    run_metrics = data['run_metrics']

    def normalise_finite_endpoints(v):
        f = v.dropna()
        a, b = (f.iloc[0], f.iloc[-1])
        return (v - b) / (a - b) if abs(a - b) > 1e-12 else v * np.nan
    run_metrics['p_h1w'] = run_metrics.groupby(['latent_dim', 'seed']).h1w.transform(normalise_finite_endpoints)
    preservation_medians = run_metrics[run_metrics.latent_dim >= 4].groupby('beta_d').median(numeric_only=True)
    with matplotlib.rc_context(plot_style):
        fig, ax = plt.subplots(figsize=(11, 5.4))
        ax.plot(beta_positions, preservation_medians.p_auc_cancer, color=NAVYW, lw=4, marker='o', label='AUC on z (cancer type)', zorder=5)
        ax.plot(beta_positions, preservation_medians.rate_rel, color=NAVYW, lw=1.6, ls='--', label='rate (share of start)')
        for col, c, l in [('trust', BLUEW, 'Trustworthiness'), ('steadiness', ORANGEW, 'Steadiness'), ('lcmc', GREENW, 'LCMC'), ('continuity', GREYW, 'Continuity'), ('h1w', '#C0392B', 'H1 Wasserstein'), ('tc', '#8E44AD', 'Total correlation'), ('coverage', '#16A085', 'Coverage')]:
            ax.plot(beta_positions, preservation_medians['p_' + col], color=c, lw=2.4, marker='.', label=l, ls='-' if col not in ('tc', 'coverage') else (0, (5, 2)))
        format_beta_axis(ax)
        ax.set_ylabel('progress: 1 = start, 0 = collapse')
        ax.set_ylim(-0.08, 1.12)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.3), frameon=False, ncol=4)
        fig.tight_layout()
        fig.savefig(output_dir / '03b_course_tc_coverage.png', dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    evs = {c: first_lasting_drop(preservation_medians['p_' + c].fillna(0).values) for c in ['auc_cancer', 'tc', 'coverage', 'lcmc', 'trust', 'steadiness', 'continuity', 'h1w']}
    plot_summary['w2'] = {k: beta_labels[int(v)] if v == v else '–' for k, v in evs.items()}
    cov_dir = 'falls' if run_metrics.groupby(['latent_dim', 'seed']).coverage.apply(lambda v: v.iloc[-1] < v.iloc[0]).mean() > 0.5 else 'rises'
    plot_summary['w2']['coverage_direction'] = cov_dir



In [ ]:
plot_preservation_course(data, OUTPUT)
show_figures('03b_course_tc_coverage.png')


## Folie 28 — 19_collapse_measures.png


In [ ]:
def plot_collapse_measures(data, output_dir):
    BLUEW = data['BLUEW']
    GREENW = data['GREENW']
    NAVYW = data['NAVYW']
    ORANGEW = data['ORANGEW']
    REDW = data['REDW']
    beta_grid = data['beta_grid']
    beta_labels = data['beta_labels']
    beta_positions = data['beta_positions']
    first_lasting_drop = data['first_lasting_drop']
    format_beta = data['format_beta']
    format_beta_axis = data['format_beta_axis']
    plot_style = data['plot_style']
    plot_summary = data['plot_summary']
    results_dir = data['results_dir']
    run_metrics = data['run_metrics']
    W = run_metrics[run_metrics.latent_dim >= 4].copy()
    W['au_frac'] = W.au / W.latent_dim
    W['rate_dim'] = W.rate / W.latent_dim
    W['zdim_frac'] = W.zdim / W.latent_dim
    W['erank_frac'] = W.erank / W.latent_dim
    G1 = W.groupby('beta_d')
    collapse_medians, q1, q3 = (G1.median(numeric_only=True), G1.quantile(0.25, numeric_only=True), G1.quantile(0.75, numeric_only=True))
    ev = lambda col: first_lasting_drop(collapse_medians[col].values)
    ec, es = (ev('p_auc_cancer'), ev('p_auc_sex'))
    B = pd.read_csv(results_dir / '30_09_B_rate_korrelation.csv').set_index('Metrik')
    rho_zdim = float(B.loc['dimensionality_z_mle', 'rho_rate'])
    rho_er = float(B.loc['dimensionality_mu_effective_rank', 'rho_rate'])
    with matplotlib.rc_context(plot_style):
        fig, ax = plt.subplots(1, 3, figsize=(20, 6.2))

        def band(a, col, c, l, ls='-'):
            a.fill_between(beta_positions, q1[col], q3[col], color=c, alpha=0.15, lw=0)
            a.plot(beta_positions, collapse_medians[col], color=c, lw=3, marker='o', ls=ls, label=l)
        band(ax[0], 'au_frac', BLUEW, 'active units (share of d)')
        band(ax[0], 'collapsed', REDW, 'collapsed dims, ε-δ (share of d)')
        ax[0].set_title('Active units and ε-δ collapse')
        ax[0].set_ylim(-0.05, 1.08)
        band(ax[1], 'rate_dim', NAVYW, 'rate per dimension')
        ax[1].set_yscale('log')
        ax[1].set_title('Rate per dimension [nats]')
        band(ax[2], 'zdim_frac', GREENW, f'intrinsic dim. of z / d (ρ with rate {rho_zdim:+.2f})')
        band(ax[2], 'erank_frac', ORANGEW, f'effective rank / d (ρ with rate {rho_er:+.2f})')
        ax[2].set_title('Dimensionality')
        ax[2].set_ylim(-0.05, 1.15)
        for a in ax:
            format_beta_axis(a)
            for e, c, l in [(ec, BLUEW, 'cancer AUC −5 %'), (es, ORANGEW, 'sex AUC −5 %')]:
                if e == e:
                    a.axvline(e, color=c, lw=1.6, ls=':')
        ax[0].legend(frameon=False, loc='center left')
        ax[2].legend(frameon=False, loc='lower left', fontsize=11)
        yl = ax[1].get_ylim()
        if ec == ec:
            ax[1].text(ec, yl[1], ' cancer −5 %', color=BLUEW, fontsize=11, va='top')
        if es == es:
            ax[1].text(es, yl[1] * 0.25, ' sex −5 %', color=ORANGEW, fontsize=11, va='top')
        fig.tight_layout()
        fig.savefig(output_dir / '19_collapse_measures.png', dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    first_inactive = next((beta_grid[i] for i in range(len(beta_grid)) if collapse_medians.au_frac.iloc[i] < 1), np.nan)
    all_collapsed = next((beta_grid[i] for i in range(len(beta_grid)) if collapse_medians.collapsed.iloc[i] >= 1), np.nan)
    lab_ = lambda i: beta_labels[int(i)] if i == i else '–'
    plot_summary['w1'] = {'first_inactive': format_beta(first_inactive) if first_inactive == first_inactive else '–', 'all_collapsed': format_beta(all_collapsed) if all_collapsed == all_collapsed else '–', 'ev_cancer': lab_(ec), 'ev_sex': lab_(es), 'rho_zdim': rho_zdim, 'rho_erank': rho_er, 'au_at_cancer': float(collapse_medians.au_frac.iloc[int(ec)]) if ec == ec else None, 'au_at_sex': float(collapse_medians.au_frac.iloc[int(es)]) if es == es else None}



In [ ]:
plot_collapse_measures(data, OUTPUT)
show_figures('19_collapse_measures.png')


## Folie 29 — 05_rate_trackers.png


In [ ]:
def plot_overview_rate(data, output_dir):
    results_dir = data['results_dir']
    run_metrics = data['run_metrics']
    import matplotlib as mpl
    output_dir.mkdir(exist_ok=True)
    NAVY, BLUE, ORANGE, GREY, GREEN, RED, LIGHT = ('#1B2430', '#2F6DB5', '#D9822B', '#9AA6B5', '#2E8B57', '#C0392B', '#DCE3EC')
    overview_plot_style = {'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 15, 'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 12, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#E6E9EE', 'axes.edgecolor': '#5E6977', 'font.family': 'DejaVu Sans'}
    NICE = {'steadiness': 'Steadiness', 'scale_normalized_stress': 'Stress (scale-norm.)', 'non_metric_stress': 'Stress (non-metric)', 'trustworthiness': 'Trustworthiness', 'continuity': 'Continuity', 'lcmc': 'LCMC', 'mrre_false': 'MRRE', 'mrre_missing': 'MRRE (missing)', 'G:edge_jaccard(x↔μ)': 'Edge overlap', 'G:clustering': 'Graph clustering', 'T:h1_normalized_wasserstein': 'H1 Wasserstein', 'T:landscape_h1_l2': 'H1 landscape', 'effective_rank': 'Effective rank', 'effective_rank_fraction': 'Effective rank (frac.)', 'dimensionality_z_mle': 'Intrinsic dim. of z', 'info_total_correlation_autoencodix': 'Total correlation', 'G:hubness_skewness': 'Hubness skewness', 'procrustes': 'Procrustes', 'G:transitivity': 'Transitivity', 'neighbor_dissimilarity': 'Neighbor dissimilarity', 'highest rate': 'Highest rate', 'rate': 'Rate'}
    nice = lambda s: NICE.get(s, s.replace('G:', '').replace('T:', '').replace('_', ' '))

    def save(fig, name):
        f = output_dir / f'{name}.png'
        fig.savefig(f, dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return f
    task_scores = run_metrics.rename(columns={'auc_cancer': 'AUC_valid', 'auc_sex': 'sex_mixed_z'})
    GRID = sorted(task_scores.beta_d.unique())
    SUP = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta_short(v):
        if v != v:
            return '–'
        if v <= 0:
            return '0'
        m, e = f'{v:.2e}'.split('e')
        return f'{float(m):.2g}·10{str(int(e)).translate(SUP)}'
    gridlab = [format_beta_short(b) for b in GRID]
    B_ = pd.read_csv(results_dir / '30_09_B_rate_korrelation.csv').set_index('Metrik')
    FIGS = {}
    short = lambda c: c.replace('geometry_mu_', '').replace('graph_latent_', 'G:').replace('_zadu', '').replace('_k20', '').replace('dimensionality_mu_', '').replace('topology_', 'T:')
    with mpl.rc_context(overview_plot_style):
        b = B_[B_.q_rate < 0.05].assign(a=lambda t: t.rho_rate.abs()).sort_values('a', ascending=False).head(8).iloc[::-1]
        fig, ax = plt.subplots(figsize=(8.6, 4.4))
        ax.barh([nice(short(c)) for c in b.index], b.rho_rate, color=[RED if v < 0 else NAVY for v in b.rho_rate])
        ax.axvline(0, color=NAVY, lw=1)
        ax.set_xlim(-1.05, 1.05)
        ax.set_xlabel('Spearman ρ with the rate (median over trajectories)')
        ax.grid(axis='y', visible=False)
        FIGS['rate'] = save(fig, '05_rate_trackers')


In [ ]:
plot_overview_rate(data, OUTPUT)
show_figures('05_rate_trackers.png')


## Folie 30 — 06_timeline.png


In [ ]:
def plot_overview_timeline(data, output_dir):
    results_dir = data['results_dir']
    run_metrics = data['run_metrics']
    sweep_dir = data['sweep_dir']
    import matplotlib as mpl
    output_dir.mkdir(exist_ok=True)
    NAVY, BLUE, ORANGE, GREY, GREEN, RED, LIGHT = ('#1B2430', '#2F6DB5', '#D9822B', '#9AA6B5', '#2E8B57', '#C0392B', '#DCE3EC')
    overview_plot_style = {'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 15, 'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 12, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#E6E9EE', 'axes.edgecolor': '#5E6977', 'font.family': 'DejaVu Sans'}

    def save(fig, name):
        f = output_dir / f'{name}.png'
        fig.savefig(f, dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return f
    indexed_metrics = pd.read_csv(sweep_dir / 'experiment_metrics.csv').set_index('run_name')
    task_scores = run_metrics.rename(columns={'auc_cancer': 'AUC_valid', 'auc_sex': 'sex_mixed_z'})
    GRID = sorted(task_scores.beta_d.unique())
    pos_of = lambda v: np.nan if v != v else 0.0 if v <= 0 else float(np.interp(np.log10(v), np.log10(GRID[1:]), np.arange(1, len(GRID))))
    SUP = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta_short(v):
        if v != v:
            return '–'
        if v <= 0:
            return '0'
        m, e = f'{v:.2e}'.split('e')
        return f'{float(m):.2g}·10{str(int(e)).translate(SUP)}'
    gridlab = [format_beta_short(b) for b in GRID]
    class_events = pd.read_csv(results_dir / '30_09_G13_klassentod_z.csv')
    FIGS = {}
    overview_summary = {}
    with mpl.rc_context(overview_plot_style):
        au = indexed_metrics['posterior_active_units_count_eps_0p01']
        ev = []
        for (dd, ss), g_ in task_scores[task_scores.latent_dim.ge(4)].groupby(['latent_dim', 'seed']):
            g_ = g_.sort_values('beta_d')
            bd = g_.beta_d.to_numpy()
            a_ = g_.run_name.map(au).to_numpy()

            def first(mask):
                return bd[np.argmax(mask)] if mask.any() else np.nan
            ev.append(dict(sex=first(1 - g_.sex_mixed_z.to_numpy() / g_.sex_mixed_z.iloc[0] > 0.02), cancer=first(1 - g_.AUC_valid.to_numpy() / g_.AUC_valid.iloc[0] > 0.02), inactive=first(a_ < dd), collapse=first(a_ == 0)))
        ev = pd.DataFrame(ev).median()
        ev['classes'] = class_events[class_events.d.ge(4) & class_events.cls.ne('sex')].death.median()
        overview_summary.update({f'ev_{k}': v for k, v in ev.items()})
        EVL = [('sex', 'sex AUC −2 %', GREEN), ('cancer', 'cancer-type AUC −2 %', BLUE), ('inactive', 'first inactive unit', ORANGE), ('classes', 'cancer types at chance', RED), ('collapse', 'full collapse', NAVY)]
        fig, ax = plt.subplots(figsize=(11, 2.9))
        ax.axhline(0, color=GREY, lw=2)
        for k, (key, lab, col) in enumerate(EVL):
            x = pos_of(ev[key])
            ax.plot(x, 0, 'o', ms=16, color=col)
            ax.annotate(f'{lab}\nβ/d ≈ {format_beta_short(ev[key])}', (x, 0), xytext=(0, 32 if k % 2 == 0 else -58), textcoords='offset points', ha='center', fontsize=13, color=col)
        ax.set_xticks(range(len(GRID)))
        ax.set_xticklabels(gridlab, fontsize=12)
        ax.set_yticks([])
        ax.set_ylim(-1, 1)
        ax.grid(False)
        ax.spines['left'].set_visible(False)
        ax.set_xlabel('β/d (compression →)')
        ax.set_xlim(-0.5, len(GRID) - 0.5)
        FIGS['timeline'] = save(fig, '06_timeline')


In [ ]:
plot_overview_timeline(data, OUTPUT)
show_figures('06_timeline.png')


## Folie 32 — 33_stop_course.png


In [ ]:
def plot_stop_curves(data, output_dir):
    """Original illustrative stopping thresholds and curves."""
    BLUET = data['BLUET']
    GREENT = data['GREENT']
    NAVYT = data['NAVYT']
    ORANGET = data['ORANGET']
    REDT = data['REDT']
    TEALT = data['TEALT']
    analysis_metrics = data['analysis_metrics']
    analysis_plot_style = data['analysis_plot_style']
    analysis_summary = data['analysis_summary']
    format_beta_label = data['format_beta_label']
    rate_correlations = data['rate_correlations']
    stop_metrics = {'geometry_mu_trustworthiness_zadu_k20': ('Trustworthiness', NAVYT),
        'geometry_mu_lcmc_zadu_k20': ('LCMC', GREENT),
        'geometry_mu_steadiness_zadu_k20': ('Steadiness', ORANGET)}
    metric_directions = {c: float(np.sign(rate_correlations.get(c, 1)) or 1) for c in stop_metrics}
    stop_states = analysis_metrics[analysis_metrics.latent_dim.ge(4) & analysis_metrics.beta.gt(0)].sort_values(['latent_dim',
        'seed', 'beta_d']).copy()
    for c in stop_metrics:
        stop_states['rel_' + c] = stop_states.groupby(['latent_dim',
            'seed'])[c].transform(lambda v: v / np.nanmean(v.iloc[:2]) if metric_directions[c] > 0 else np.nanmean(v.iloc[:2]) / v)
    stop_beta_positions = np.arange(stop_states.beta_d.nunique())
    stop_beta_labels = [format_beta_label(v) for v in sorted(stop_states.beta_d.unique())]
    for t in ['auc_cancer', 'auc_sex_all']:
        stop_states['rel_' + t] = stop_states.groupby(['latent_dim',
            'seed'])[t].transform(lambda v: v / np.nanmean(v.iloc[:2]))
    stop_states['rate_rel'] = stop_states.R / stop_states.groupby(['latent_dim',
        'seed']).R.transform('first')
    rows = []
    for (dd, ss), g in stop_states.groupby(['latent_dim', 'seed']):
        g = g.reset_index(drop=True)

        def stop(mask):
            f = np.flatnonzero(mask)
            return max(f[0] - 1 if len(f) else len(g) - 1, 0)
        loss = {t: 1 - g['rel_' + t].to_numpy() for t in ['auc_cancer', 'auc_sex_all']}
        for t in loss:
            i = stop(loss[t] > 0.02)
            rows.append(dict(rule=f"true point ({('cancer' if t == 'auc_cancer' else 'sex')})",
                d=dd, seed=ss, i=i, beta_d=g.beta_d[i], rate_saved=1 - g.rate_rel[i],
                loss_cancer=loss['auc_cancer'][i], loss_sex=loss['auc_sex_all'][i]))
        for c, (nm, _) in stop_metrics.items():
            rel = np.nan_to_num(g['rel_' + c].to_numpy(), nan=-np.inf)
            i = stop(rel < 0.99)
            rows.append(dict(rule=f'{nm} −1 %', d=dd, seed=ss, i=i, beta_d=g.beta_d[i],
                rate_saved=1 - g.rate_rel[i], loss_cancer=loss['auc_cancer'][i],
                loss_sex=loss['auc_sex_all'][i]))
        for kind, vals, grid in [('rate < r', g.R.to_numpy(), np.logspace(-1, 2.5, 36)),
            ('rate/d < r', (g.R / g.latent_dim).to_numpy(), np.logspace(-2, 1.5, 36)),
            ('rate/start < r', g.rate_rel.to_numpy(), np.logspace(-3, 0, 31))]:
            for r_ in grid:
                i = stop(vals < r_)
                rows.append(dict(rule=kind, thr=r_, d=dd, seed=ss, i=i, beta_d=g.beta_d[i],
                    rate_saved=1 - g.rate_rel[i], loss_cancer=loss['auc_cancer'][i],
                    loss_sex=loss['auc_sex_all'][i]))
    stop_results = pd.DataFrame(rows)
    stop_results.to_csv(output_dir / '30_09_T3_stopp_verlauf.csv', index=False)
    RR3 = stop_results[stop_results.rule.str.startswith('rate')].groupby(['rule',
        'thr']).agg(rate_saved=('rate_saved', lambda v: 100 * v.median()),
        too_far_cancer=('loss_cancer', lambda v: 100 * (v > 0.02).mean()), too_far_sex=('loss_sex',
        lambda v: 100 * (v > 0.02).mean())).reset_index()
    BESTR = pd.concat([RR3[RR3['too_far_' + t] <= 10].sort_values('rate_saved').groupby('rule').tail(1).assign(task=t) for t in ['cancer',
        'sex']])
    stop_results = stop_results[~stop_results.rule.str.startswith('rate')]
    stop_summary = stop_results.groupby('rule').agg(stop_idx=('i', 'median'),
        rate_saved=('rate_saved', lambda v: 100 * v.median()), too_far_cancer=('loss_cancer',
        lambda v: 100 * (v > 0.02).mean()), too_far_sex=('loss_sex',
        lambda v: 100 * (v > 0.02).mean()), loss_cancer=('loss_cancer',
        lambda v: 100 * v.median()), loss_sex=('loss_sex', lambda v: 100 * v.median()))
    stop_groups = stop_states.groupby('beta_d')
    stop_medians, stop_lower_quartiles, stop_upper_quartiles = (stop_groups.median(numeric_only=True),
        stop_groups.quantile(0.25, numeric_only=True), stop_groups.quantile(0.75,
        numeric_only=True))
    with matplotlib.rc_context(analysis_plot_style):
        fig, ax = plt.subplots(1, 2, figsize=(18, 6.4))
        for c, (nm, colr) in stop_metrics.items():
            ax[0].fill_between(stop_beta_positions, stop_lower_quartiles['rel_' + c],
                stop_upper_quartiles['rel_' + c], color=colr, alpha=0.1, lw=0)
            ax[0].plot(stop_beta_positions, stop_medians['rel_' + c], color=colr, lw=2.8,
                marker='o', label=nm)
        ax[0].axhline(0.99, color=REDT, ls='--', lw=1.6)
        ax[0].text(stop_beta_positions[-1], 0.99, '−1 % ', color=REDT, ha='right', va='bottom',
            fontsize=12)
        ax[0].set_ylim(0.8, 1.015)
        ax[0].set_title('Label-free: metric relative to its start', loc='left')
        ax[0].set_ylabel('metric / start value')
        ax[0].legend(frameon=False, loc='lower left')
        for t, colr, nm in [('auc_cancer', BLUET, 'cancer-type AUC'), ('auc_sex_all', TEALT,
            'sex AUC (mixed types)')]:
            ax[1].fill_between(stop_beta_positions, stop_lower_quartiles['rel_' + t],
                stop_upper_quartiles['rel_' + t], color=colr, alpha=0.12, lw=0)
            ax[1].plot(stop_beta_positions, stop_medians['rel_' + t], color=colr, lw=3, marker='o',
                label=nm)
        ax[1].axhline(0.98, color=REDT, ls='--', lw=1.6)
        ax[1].text(stop_beta_positions[-1], 0.98, '−2 % ', color=REDT, ha='right', va='bottom',
            fontsize=12)
        ax[1].set_ylim(0.5, 1.06)
        ax[1].set_title('AUC relative to its start', loc='left')
        ax[1].set_ylabel('AUC / start value')
        ax[1].legend(frameon=False, loc='lower left')
        for a in ax:
            a.set_xticks(stop_beta_positions)
            a.set_xticklabels(stop_beta_labels, rotation=45)
            a.set_xlabel('β/d (compression →)')
        fig.tight_layout()
        fig.savefig(output_dir / '33_stop_course.png', dpi=200, bbox_inches='tight',
            facecolor='white')
        plt.close(fig)
    analysis_summary['t3'] = stop_summary.round(2).to_dict('index')
    analysis_summary['t3_rate_rules'] = BESTR.round(3).to_dict('records')


In [ ]:
plot_stop_curves(data, OUTPUT)
show_figures('33_stop_course.png')


## Folie 33 — 22_warn_strip.png


In [ ]:
def plot_warning_strip(data, output_dir):
    """Original d=8, seed=17 warning/stop strip."""
    BLUEW = data['BLUEW']
    GREENW = data['GREENW']
    LIGHTW = data['LIGHTW']
    NAVYW = data['NAVYW']
    beta_grid = data['beta_grid']
    beta_labels = data['beta_labels']
    clinical_metadata = data['clinical_metadata']
    first_lasting_drop = data['first_lasting_drop']
    plot_style = data['plot_style']
    plot_summary = data['plot_summary']
    run_metrics = data['run_metrics']
    sweep_dir = data['sweep_dir']
    latent_dim, seed = (8, 17)
    shown_beta_values = [0, 1.25e-05, 5e-05, 0.0001, 0.00025, 0.001, 0.004]
    T = run_metrics[(run_metrics.latent_dim == latent_dim) & (run_metrics.seed == seed)].reset_index(drop=True)
    tr0 = T.trust.iloc[0]
    stop_i = next((i - 1 for i in range(len(T)) if (T.trust.iloc[i:] < 0.99 * tr0).all()), np.nan)
    warning_events = {'TC warns': first_lasting_drop(T.p_tc),
        'LCMC warns': first_lasting_drop(T.p_lcmc), 'stop (trustworthiness −1 %)': stop_i,
        'cancer AUC −5 %': first_lasting_drop(T.p_auc_cancer),
        'sex AUC −5 %': first_lasting_drop(T.p_auc_sex)}
    event_colors = {'TC warns': '#8E44AD', 'LCMC warns': BLUEW,
        'stop (trustworthiness −1 %)': GREENW, 'cancer AUC −5 %': NAVYW, 'sex AUC −5 %': '#00A6A6'}
    shown_beta_indices = [int(np.argmin(np.abs(np.array(beta_grid) - b))) for b in shown_beta_values]

    def col_of(i):
        if i != i:
            return None
        return next((k for k, pi in enumerate(shown_beta_indices) if pi >= i), None)
    cancer_types = sorted(clinical_metadata.CANCER_TYPE_ACRONYM.astype(str).unique())
    cancer_palette = plt.get_cmap('tab10').colors
    mixed_cancer_types = ['COAD', 'LUAD', 'LUSC', 'READ']

    def stars(ax, E, lab, cmap):
        for k, g in E.groupby(lab.values):
            if k in cmap:
                ax.scatter(g[0].mean(), g[1].mean(), marker='*', s=520, color=cmap[k],
                    edgecolor='black', linewidth=1.2, zorder=5)
    with matplotlib.rc_context(plot_style):
        fig, axes = plt.subplots(2, len(shown_beta_values), figsize=(3.3 * len(shown_beta_values),
            8.6))
        for j, pi in enumerate(shown_beta_indices):
            r = T.iloc[pi]
            E = pd.read_csv(sweep_dir / 'plots' / r.run_name / '2D_embedding_z_all.csv',
                index_col=0)
            E.index = E.index.astype(str)
            E.columns = [0, 1]
            ct = clinical_metadata.CANCER_TYPE_ACRONYM.reindex(E.index).astype(str)
            sx = clinical_metadata.SEX.reindex(E.index).astype(str)
            a = axes[0, j]
            a.scatter(E[0], E[1], c=[cancer_palette[cancer_types.index(c)] for c in ct], s=2,
                alpha=0.55, linewidths=0)
            stars(a, E, ct, {c: cancer_palette[i] for i, c in enumerate(cancer_types)})
            a.set_title(f'β/d = {beta_labels[pi]}\ncancer-type AUC {r.auc_cancer:.2f}', fontsize=12)
            mk = (ct.isin(mixed_cancer_types) & sx.isin(['Female', 'Male'])).values
            b = axes[1, j]
            unk = (ct.isin(mixed_cancer_types) & sx.eq('unknown')).values
            b.scatter(E[0][~mk & ~unk], E[1][~mk & ~unk], c=LIGHTW, s=2, linewidths=0)
            b.scatter(E[0][unk], E[1][unk], c='#9AA6B5', s=4, linewidths=0)
            b.scatter(E[0][mk], E[1][mk], c=list(sx[mk].map({'Female': '#00A6A6',
                'Male': '#1A1A1A'})), s=3, alpha=0.75, linewidths=0)
            for (t_, k), g in E[mk].groupby([ct[mk].values, sx[mk].values]):
                b.scatter(g[0].mean(), g[1].mean(), marker='*', s=420, color={'Female': '#00A6A6',
                    'Male': '#1A1A1A'}[k], edgecolor='black', linewidth=1.1, zorder=5)
            b.set_title(f'sex AUC {r.auc_sex:.2f}', fontsize=12)
            for x_ in (a, b):
                x_.set_xticks([])
                x_.set_yticks([])
                x_.grid(False)
                for sp in x_.spines.values():
                    sp.set_visible(True)
                    sp.set_color('#C9CED6')
        axes[0, 0].set_ylabel('TASK 1\ncancer type', fontsize=15, color='#2F6DB5',
            fontweight='bold')
        axes[1, 0].set_ylabel('TASK 2\nsex (mixed types)', fontsize=15, color='#00A6A6',
            fontweight='bold')
        stack = {}
        for name, i in warning_events.items():
            j = col_of(i)
            if j is None:
                continue
            k = stack.get(j, 0)
            stack[j] = k + 1
            axes[0, j].text(0.5, 1.38 + 0.16 * k, name, transform=axes[0, j].transAxes, ha='center',
                va='bottom', fontsize=14, color='white', fontweight='bold',
                bbox=dict(boxstyle='round,pad=.3', fc=event_colors[name], ec='none'))
            for x_ in (axes[0, j], axes[1, j]):
                for sp in x_.spines.values():
                    sp.set_color(event_colors[name])
                    sp.set_linewidth(2.5)
        H4 = [plt.Line2D([], [], marker='o', ls='', color=cancer_palette[i], label=c) for i,
            c in enumerate(cancer_types)] + [plt.Line2D([], [], marker='o', ls='', color=c,
            label=l) for l, c in [('female', '#00A6A6'), ('male', '#1A1A1A'), ('sex unknown',
            '#9AA6B5'), ('single-sex types', '#DCE3EC')]]
        fig.legend(handles=H4, loc='lower center', ncol=12, frameon=False, fontsize=12,
            bbox_to_anchor=(0.5, -0.04))
        fig.tight_layout(rect=(0, 0, 1, 0.93))
        fig.savefig(output_dir / '22_warn_strip.png', dpi=170, bbox_inches='tight',
            facecolor='white')
        plt.close(fig)
    plot_summary['w4'] = {'d': latent_dim, 'seed': seed,
        'events': {k: beta_labels[int(v)] if v == v else '–' for k, v in warning_events.items()}}


In [ ]:
plot_warning_strip(data, OUTPUT)
show_figures('22_warn_strip.png')


## Folie 34 — 11_edge_cancer_sex.png


In [ ]:
def plot_overview_edge(data, output_dir):
    run_metrics = data['run_metrics']
    import matplotlib as mpl
    output_dir.mkdir(exist_ok=True)
    NAVY, BLUE, ORANGE, GREY, GREEN, RED, LIGHT = ('#1B2430', '#2F6DB5', '#D9822B', '#9AA6B5', '#2E8B57', '#C0392B', '#DCE3EC')
    overview_plot_style = {'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 15, 'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 12, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#E6E9EE', 'axes.edgecolor': '#5E6977', 'font.family': 'DejaVu Sans'}

    def save(fig, name):
        f = output_dir / f'{name}.png'
        fig.savefig(f, dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return f
    task_scores = run_metrics.rename(columns={'auc_cancer': 'AUC_valid', 'auc_sex': 'sex_mixed_z'})
    GRID = sorted(task_scores.beta_d.unique())
    pos_of = lambda v: np.nan if v != v else 0.0 if v <= 0 else float(np.interp(np.log10(v), np.log10(GRID[1:]), np.arange(1, len(GRID))))
    SUP = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta_short(v):
        if v != v:
            return '–'
        if v <= 0:
            return '0'
        m, e = f'{v:.2e}'.split('e')
        return f'{float(m):.2g}·10{str(int(e)).translate(SUP)}'
    gridlab = [format_beta_short(b) for b in GRID]
    FIGS = {}
    overview_summary = {}
    with mpl.rc_context(overview_plot_style):
        gg = task_scores[task_scores.latent_dim.ge(4)].groupby('beta_d')[['AUC_valid', 'sex_mixed_z']].median()
        overview_summary['sex0'], overview_summary['sex5e5'] = (gg.sex_mixed_z.iloc[0], gg.sex_mixed_z.loc[5e-05])
        overview_summary['can0'], overview_summary['can5e5'] = (gg.AUC_valid.iloc[0], gg.AUC_valid.loc[5e-05])
        fig, ax = plt.subplots(figsize=(9.6, 4.6))
        xs = [pos_of(b) for b in gg.index]
        ax.plot(xs, gg.AUC_valid, color=BLUE, lw=3.2, marker='o', label='cancer type (AUC on z)')
        ax.plot(xs, gg.sex_mixed_z, color=GREEN, lw=3.2, marker='o', label='sex, mixed cancer types (AUC on z)')
        ax.axhline(0.5, color=GREY, lw=1)
        ax.text(0.1, 0.51, 'chance', ha='left', color=GREY, fontsize=11)
        ax.set_xticks(range(len(GRID)))
        ax.set_xticklabels(gridlab, fontsize=12)
        ax.set_ylim(0.4, 1.05)
        ax.set_xlabel('β/d (compression →)')
        ax.set_ylabel('AUC (median, d ≥ 4)')
        ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2)
        FIGS['edge'] = save(fig, '11_edge_cancer_sex')


In [ ]:
plot_overview_edge(data, OUTPUT)
show_figures('11_edge_cancer_sex.png')


## Folie 35 — 12_classes.png


In [ ]:
def plot_overview_classes(data, output_dir):
    results_dir = data['results_dir']
    run_metrics = data['run_metrics']
    import matplotlib as mpl
    output_dir.mkdir(exist_ok=True)
    NAVY, BLUE, ORANGE, GREY, GREEN, RED, LIGHT = ('#1B2430', '#2F6DB5', '#D9822B', '#9AA6B5', '#2E8B57', '#C0392B', '#DCE3EC')
    overview_plot_style = {'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 15, 'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 12, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#E6E9EE', 'axes.edgecolor': '#5E6977', 'font.family': 'DejaVu Sans'}

    def save(fig, name):
        f = output_dir / f'{name}.png'
        fig.savefig(f, dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return f
    task_scores = run_metrics.rename(columns={'auc_cancer': 'AUC_valid', 'auc_sex': 'sex_mixed_z'})
    GRID = sorted(task_scores.beta_d.unique())
    pos_of = lambda v: np.nan if v != v else 0.0 if v <= 0 else float(np.interp(np.log10(v), np.log10(GRID[1:]), np.arange(1, len(GRID))))
    SUP = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')

    def format_beta_short(v):
        if v != v:
            return '–'
        if v <= 0:
            return '0'
        m, e = f'{v:.2e}'.split('e')
        return f'{float(m):.2g}·10{str(int(e)).translate(SUP)}'
    gridlab = [format_beta_short(b) for b in GRID]
    class_events = pd.read_csv(results_dir / '30_09_G13_klassentod_z.csv')
    FIGS = {}
    with mpl.rc_context(overview_plot_style):
        dt = class_events[class_events.d.ge(4)].groupby('cls')[['not_good', 'death']].median().sort_values('not_good')
        fig, ax = plt.subplots(figsize=(9.6, 4.6))
        yy = np.arange(len(dt))
        for k, (cl, r) in enumerate(dt.iterrows()):
            a_, b_ = (pos_of(r.not_good), pos_of(r.death))
            ax.plot([a_, b_], [k, k], color=LIGHT, lw=6, solid_capstyle='round')
            ax.plot(a_, k + 0.14, 'o', ms=11, color=ORANGE)
            ax.plot(b_, k - 0.14, 'D', ms=10, color=RED)
        ax.set_yticks(yy)
        ax.set_yticklabels(dt.index)
        ax.set_xticks(range(len(GRID)))
        ax.set_xticklabels(gridlab, fontsize=12)
        ax.set_xlim(-0.3, len(GRID) - 0.5)
        ax.plot([], [], 'o', color=ORANGE, label='no longer well predicted (AUC < 0.9)')
        ax.plot([], [], 'D', color=RED, label='at chance (AUC not > 0.5)')
        ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=12)
        ax.set_xlabel('β/d (compression →)')
        ax.grid(axis='y', visible=False)
        FIGS['classes'] = save(fig, '12_classes')


In [ ]:
plot_overview_classes(data, OUTPUT)
show_figures('12_classes.png')


## Folie 40 — 33_stop_course.png


In [ ]:
show_figures('33_stop_course.png')


## Vollständigkeit prüfen


In [ ]:
# Preserve the three shared plot summaries independently of slide order.
with (OUTPUT / 'wtasks.json').open('w') as handle:
    json.dump({key: data['plot_summary'][key] for key in ['w1', 'w2', 'w7']},
              handle, ensure_ascii=False, indent=1)

inventory = pd.read_parquet(RESULTS / 'presentation_diagrams/diagram_inventory.parquet')
inventory['regenerated'] = inventory.description.map(lambda filename: (OUTPUT / filename).is_file())
assert inventory.regenerated.all(), 'Mindestens ein Diagramm fehlt'
display(inventory[['slide', 'description', 'regenerated']])
